### Cartesian radiating sphere test

Flux factor $f = F_r / E$ and Eddington factor $p = P_{rr} / E$ from the last mesh snapshot along the outward $+x$ ray from the sphere center, compared with the analytic radiating-sphere solution.

In [ ]:
import os
import sys
from pathlib import Path
import glob

import matplotlib.pyplot as plt
import h5py
import numpy as np

try:
    here = Path(__file__).resolve().parent
except NameError:
    here = Path.cwd()

sys.path.append(str((here / '..' / 'plot_functions').resolve()))

import plot_functions as pf

plots_dir = here / 'plots'
plots_dir.mkdir(exist_ok=True)

Simulation and test parameters

In [ ]:
# Directory with mesh_plt*.h5 dumps (only the last dump is used below)
# directory_path = (
#     '/pscratch/sd/b/bigqke/bigqke_scratch/emu_methods_paper_simulations/bigqke_per/radiating_sphere_test/'
#     # 'radiating_sphere_test_L_cm_3.627e6_2.457e6_2.457e6_ncell_128_128_128_dts_1.60e-7_deposition_1_bc_periodic_empty_ppepc_378_stored'
#     # 'radiating_sphere_test_L_cm_3.627e6_2.457e6_2.457e6_ncell_128_128_128_dts_1.60e-7_deposition_1_bc_outflow_ppepc_92_stored'
#     'radiating_sphere_test_L_cm_3.627e6_2.457e6_2.457e6_ncell_128_128_128_dts_1.60e-7_deposition_1_bc_periodic_empty_ppepc_92_stored'
#     ''
# )
# Nx, Ny, Nz = 128, 128, 128
# r_xi, r_yi, r_zi = 45, 64, 64 

###################################
# changing cell size

directory_path = (
    '/pscratch/sd/b/bigqke/bigqke_scratch/emu_methods_paper_simulations/bigqke_per/radiating_sphere_test/'
    'radiating_sphere_test_L_cm_4.212e6_2.808e6_2.808e6_ncell_96_64_64_dts_3.66e-7_deposition_1_bc_periodic_empty_ppepc_378_stored'    
)
Nx, Ny, Nz = 96, 64, 64 # Cartesian grid size; checked against the last dump
r_xi, r_yi, r_zi = 32, 32, 32 # Sphere-center cell index

directory_path = (
    '/pscratch/sd/b/bigqke/bigqke_scratch/emu_methods_paper_simulations/bigqke_per/radiating_sphere_test/'
    'radiating_sphere_test_L_cm_4.212e6_2.808e6_2.808e6_ncell_192_128_128_dts_1.83e-7_deposition_1_bc_periodic_empty_ppepc_378_stored'    
)
Nx, Ny, Nz = 192, 128, 128 # Cartesian grid size; checked against the last dump
r_xi, r_yi, r_zi = 64, 64, 64 # Sphere-center cell index

directory_path = (
    '/pscratch/sd/b/bigqke/bigqke_scratch/emu_methods_paper_simulations/bigqke_per/radiating_sphere_test/'
    'radiating_sphere_test_L_cm_4.212e6_2.808e6_2.808e6_ncell_384_256_256_dts_9.15e-8_deposition_1_bc_periodic_empty_ppepc_378_stored'
)
Nx, Ny, Nz = 384, 256, 256 # Cartesian grid size; checked against the last dump
r_xi, r_yi, r_zi = 128, 128, 128 # Sphere-center cell index

###################################
# changing number of particles per cell

# directory_path = (
#     '/pscratch/sd/b/bigqke/bigqke_scratch/emu_methods_paper_simulations/bigqke_per/radiating_sphere_test/'
#     'radiating_sphere_test_L_cm_4.212e6_2.808e6_2.808e6_ncell_384_256_256_dts_9.15e-8_deposition_1_bc_periodic_empty_ppepc_378_stored'
#     # 'radiating_sphere_test_L_cm_4.212e6_2.808e6_2.808e6_ncell_384_256_256_dts_9.15e-8_deposition_1_bc_periodic_empty_ppepc_212_stored'
#     # 'radiating_sphere_test_L_cm_4.212e6_2.808e6_2.808e6_ncell_384_256_256_dts_9.15e-8_deposition_1_bc_periodic_empty_ppepc_92_stored'
# )
# Nx, Ny, Nz = 384, 256, 256 # Cartesian grid size; checked against the last dump
# r_xi, r_yi, r_zi = 128, 128, 128 # Sphere-center cell index

# Physics / comparison parameters (same as the 1D spherical test, except r_min_over_R).
# The 1D script cuts at 0.6; here we score the whole ray from the center outward.
R = 1.17e6          # sphere radius, cm
kappa_int_invcm = 0.00000854544806011998 # electron neutrino absorption opacity, 1/cm
kappa_ext_invcm = 1.4877457720665111e-9 # electron neutrino absorption opacity, 1/cm
kappa_R = kappa_int_invcm * R # radial optical depth used in the analytic path length
n_mu = 2000         # number of mu bins for the analytic moment integrals
tolerance = 0.07    # max allowed absolute error in f and p
r_min_over_R = 0.0  # inner radius cutoff for the error test (include all r >= 0)

# plots
limlow = None
limup = -1

In [ ]:
def print_variable(name, value):
    """Print a pasteable assignment. '-' in the run name becomes 'min'."""
    run = directory_path.split("/")[-1].replace("-", "min").replace(".", "p")
    print(f"{name}_{run} = {value}")

Read the last mesh dump and extract the outward $+x$ ray

In [ ]:
# Sort by the integer plot index in mesh_pltNNNNN.h5 so [-1] is the final dump.
filenames = sorted(
    glob.glob(os.path.join(directory_path, 'mesh_plt*.h5')),
    key=lambda x: int(os.path.basename(x).split('plt')[1].split('.')[0]),
)

if len(filenames) == 0:
    raise FileNotFoundError(f'No mesh_plt*.h5 files found in {directory_path}')

if not (0 <= r_xi < Nx and 0 <= r_yi < Ny and 0 <= r_zi < Nz):
    raise ValueError(
        f'Sphere-center cell ({r_xi}, {r_yi}, {r_zi}) is outside '
        f'grid ({Nx}, {Ny}, {Nz})'
    )

# Use only the last snapshot (plot index matches nsteps = 508 for this run).
filename = filenames[-1]

with h5py.File(filename, 'r') as hf:
    file_Nx = int(np.array(hf['Nx']))
    file_Ny = int(np.array(hf['Ny']))
    file_Nz = int(np.array(hf['Nz']))
    if (file_Nx, file_Ny, file_Nz) != (Nx, Ny, Nz):
        raise ValueError(
            f'{os.path.basename(filename)} has grid '
            f'({file_Nx}, {file_Ny}, {file_Nz}), expected ({Nx}, {Ny}, {Nz})'
        )

    # Mesh moments: N00_Re = number density; along +x, Fx and Pxx are F_r and P_rr.
    E = np.array(hf['N00_Re(1|ccm)'], dtype=float)
    Fx = np.array(hf['Fx00_Re(1|ccm)'], dtype=float)
    Pxx = np.array(hf['Pxx00_Re(1|ccm)'], dtype=float)

    if E.shape != (Nx, Ny, Nz):
        raise ValueError(
            f'{os.path.basename(filename)} field shape {E.shape} '
            f'does not match ({Nx}, {Ny}, {Nz})'
        )

    dx = float(np.array(hf['dx(cm)']))
    dy = float(np.array(hf['dy(cm)']))
    dz = float(np.array(hf['dz(cm)']))

    # Outward +x ray through the sphere-center cell (domain origin at x = 0).
    E_ray = E[r_xi:, r_yi, r_zi].copy()
    Fr_ray = Fx[r_xi:, r_yi, r_zi].copy()
    Prr_ray = Pxx[r_xi:, r_yi, r_zi].copy()

# Cell centers along x; r = 0 at the sphere-center cell center.
x_centers = (np.arange(Nx) + 0.5) * dx
r_centers = x_centers[r_xi:] - x_centers[r_xi]

print(f'Read last dump {os.path.basename(filename)} from {directory_path}')
print(f'Grid (Nx, Ny, Nz) = ({Nx}, {Ny}, {Nz})')
print(f'dx, dy, dz = {dx:.6g}, {dy:.6g}, {dz:.6g} cm')
print(f'Sphere-center cell (r_xi, r_yi, r_zi) = ({r_xi}, {r_yi}, {r_zi})')
print(f'Ray length = {len(r_centers)} cells along +x')

Using the last snapshot of the simulation to compute flux factor and Eddington factor

In [ ]:
# Non-positive density on the ray becomes NaN so empty cells drop out of the error mask.
E_ray[E_ray <= 0.0] = np.nan

# Constants cancel in the ratios: f = F_r / E, p = P_rr / E from the last dump only.
f_simulation = Fr_ray / E_ray
p_simulation = Prr_ray / E_ray

Analytic radiating-sphere solution

In [ ]:
# Analytic f and p at each cell center along the ray (same formulas as radiating_sphere_test.py).
f_analytic = np.zeros(len(r_centers))
p_analytic = np.zeros(len(r_centers))

for i, ri in enumerate(r_centers):
    if ri < R:
        # Inside: all mu in [-1, 1]; s is path length back to the surface in units of R.
        mu_arr = np.linspace(-1.0, 1.0, n_mu)
        g = np.sqrt(np.maximum(1.0 - (ri / R)**2 * (1.0 - mu_arr**2), 0.0))
        s = ri * mu_arr / R + g
    else:
        # Outside: only mu > sqrt(1 - (R/r)^2) intersects the sphere; s = 2g is the chord.
        mu_c = np.sqrt(1.0 - (R / ri)**2)
        mu_arr = np.linspace(mu_c, 1.0, n_mu)
        g = np.sqrt(np.maximum(1.0 - (ri / R)**2 * (1.0 - mu_arr**2), 0.0))
        s = 2 * g

    # Specific intensity / B = 1 - exp(-kappa_R * s); B and azimuth cancel in the ratios.
    F_mu = 1.0 - np.exp(-kappa_R * s)

    E_int = np.trapezoid(F_mu, mu_arr)
    f_analytic[i] = np.trapezoid(mu_arr * F_mu, mu_arr) / E_int
    p_analytic[i] = np.trapezoid(mu_arr**2 * F_mu, mu_arr) / E_int

Error estimate (last snapshot vs analytic)

In [ ]:
# Compare last-snapshot f and p to the analytic solution where E is finite and r is outside the cutoff.
valid = np.isfinite(f_simulation) & (r_centers >= r_min_over_R * R)

f_error = np.abs(f_simulation[valid] - f_analytic[valid])
p_error = np.abs(p_simulation[valid] - p_analytic[valid])

f_max_error = np.max(f_error)
p_max_error = np.max(p_error)

r_valid = r_centers[valid]

print(
    f'Last dump {os.path.basename(filename)} '
    f'for the region r/R >= {r_min_over_R}'
)
print(
    f'Max error in f: {f_max_error}  '
    f'at r/R = {r_valid[np.argmax(f_error)] / R:.4f}'
)
print(
    f'Max error in p: {p_max_error}  '
    f'at r/R = {r_valid[np.argmax(p_error)] / R:.4f}'
)

if f_max_error < tolerance and p_max_error < tolerance:
    print(f'SUCCESS (errors below tolerance {tolerance})')
else:
    print(f'FAIL (errors above tolerance {tolerance})')

Plot simulation vs analytic $f$ and $p$

In [ ]:
# Profile of last-snapshot and analytic f, p versus r/R along the +x ray.
r_over_R = r_centers / R

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(r_over_R[limlow:limup], f_simulation[limlow:limup], color='C0', linestyle='dotted', linewidth=4, label=r'$\hat{\mathrm{f}}_{\mathrm{sim}}$')
ax.plot(r_over_R[limlow:limup], f_analytic[limlow:limup], color='C0', linestyle='solid', alpha=0.5, linewidth=4, label=r'$\hat{\mathrm{f}}_{\mathrm{analytic}}$')
ax.plot(r_over_R[limlow:limup], p_simulation[limlow:limup], color='C1', linestyle='dotted', linewidth=4, label=r'$\hat{\mathrm{p}}_{\mathrm{sim}}$')
ax.plot(r_over_R[limlow:limup], p_analytic[limlow:limup], color='C1', linestyle='solid', alpha=0.5, linewidth=4, label=r'$\hat{\mathrm{p}}_{\mathrm{analytic}}$')
ax.axvline(1.0, color='k', linestyle=':', linewidth=1.5, label=r'$r = R$')
ax.axvline(r_min_over_R, color='gray', linestyle=':', linewidth=1.5,
           label=rf'$r/R = {r_min_over_R}$')
ax.set_xlabel(r'$r / R$')
ax.set_ylabel(r'$f,\, p$')
ax.set_xlim(None, None)
leg = ax.legend(framealpha=0.0, ncol=1, fontsize=16)
pf.apply_custom_settings(ax, leg, False)
outfile = plots_dir / f"{directory_path.rstrip('/').split('/')[-1]}_f_p_vs_r.pdf"
fig.savefig(outfile, bbox_inches='tight')
plt.show()
plt.close(fig)
print(f'Saved {outfile}')

In [ ]:
print_variable("r_over_R", [float(x) for x in r_over_R[limlow:limup]])
print_variable("f_simulation", [float(x) for x in f_simulation[limlow:limup]])
print_variable("f_analytic", [float(x) for x in f_analytic[limlow:limup]])
print_variable("p_simulation", [float(x) for x in p_simulation[limlow:limup]])
print_variable("p_analytic", [float(x) for x in p_analytic[limlow:limup]])

Absolute error versus $r/R$

In [ ]:
# Absolute |f_sim - f_analytic| and |p_sim - p_analytic| on the valid ray points, with the tolerance line.
fig, ax = plt.subplots(figsize=(10, 6))
# ax.plot(r_over_R[valid][limlow:limup], f_error[limlow:limup], color='C0', linewidth=4, label=r'$|f_{\mathrm{sim}} - f_{\mathrm{analytic}}|$')
# ax.plot(r_over_R[valid][limlow:limup], p_error[limlow:limup], color='C1', linewidth=4, label=r'$|p_{\mathrm{sim}} - p_{\mathrm{analytic}}|$')
ax.plot(r_over_R[valid][limlow:limup], np.log(f_error[limlow:limup]), color='C0', linewidth=4, label=r'$\log\,|f_{\mathrm{sim}} - f_{\mathrm{analytic}}|$')
ax.plot(r_over_R[valid][limlow:limup], np.log(p_error[limlow:limup]), color='C1', linewidth=4, label=r'$\log\,|p_{\mathrm{sim}} - p_{\mathrm{analytic}}|$')

print(f'r_over_R = {list(r_over_R[valid][limlow:limup])}')
print(f'f_error = {list(f_error[limlow:limup])}')
print(f'p_error = {list(p_error[valid][limlow:limup])}')

ax.axhline(np.log(tolerance), color='k', linestyle='--', linewidth=1.5,
           label=rf'tolerance $= {tolerance}$')
ax.axvline(1.0, color='k', linestyle=':', linewidth=1.5, label=r'$r = R$')
ax.set_xlabel(r'$r / R$')
ax.set_ylabel(r'$\log(\, \mathrm{Absolute\,error}\,)$')
ax.set_ylim(-10, -1)
ax.set_xlim(0.0, 2.0)
leg = ax.legend(framealpha=0.0, ncol=1, fontsize=16)
pf.apply_custom_settings(ax, leg, False)
outfile = plots_dir / f"{directory_path.rstrip('/').split('/')[-1]}_f_p_error_vs_r.pdf"
fig.savefig(outfile, bbox_inches='tight')
plt.show()
plt.close(fig)
print(f'Saved {outfile}')

In [ ]:
L_1_error_in_f = np.sum(f_error[limlow:limup])/len(f_error[limlow:limup])
L_1_error_in_p = np.sum(p_error[limlow:limup])/len(p_error[limlow:limup])

L_2_error_in_f = np.sqrt(np.sum(f_error[limlow:limup]**2)/len(f_error[limlow:limup]))
L_2_error_in_p = np.sqrt(np.sum(p_error[limlow:limup]**2)/len(p_error[limlow:limup]))

L_inf_error_in_f = np.max(f_error[limlow:limup])
L_inf_error_in_p = np.max(p_error[limlow:limup])

print_variable("L_1_error_in_f", L_1_error_in_f)
print_variable("L_1_error_in_p", L_1_error_in_p)
print_variable("L_2_error_in_f", L_2_error_in_f)
print_variable("L_2_error_in_p", L_2_error_in_p)
print_variable("L_inf_error_in_f", L_inf_error_in_f)
print_variable("L_inf_error_in_p", L_inf_error_in_p)

# Convergence on dx

In [ ]:
L_1_error_in_f_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_96_64_64_dts_3p66emin7_deposition_1_bc_periodic_empty_ppepc_378_stored = 0.022829117848479786
L_1_error_in_p_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_96_64_64_dts_3p66emin7_deposition_1_bc_periodic_empty_ppepc_378_stored = 0.025907883461710565
L_2_error_in_f_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_96_64_64_dts_3p66emin7_deposition_1_bc_periodic_empty_ppepc_378_stored = 0.042687194903180056
L_2_error_in_p_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_96_64_64_dts_3p66emin7_deposition_1_bc_periodic_empty_ppepc_378_stored = 0.03811457005139076
L_inf_error_in_f_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_96_64_64_dts_3p66emin7_deposition_1_bc_periodic_empty_ppepc_378_stored = 0.23996300108160273
L_inf_error_in_p_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_96_64_64_dts_3p66emin7_deposition_1_bc_periodic_empty_ppepc_378_stored = 0.09678426791956535

L_1_error_in_f_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_192_128_128_dts_1p83emin7_deposition_1_bc_periodic_empty_ppepc_378_stored = 0.012216167062370763
L_1_error_in_p_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_192_128_128_dts_1p83emin7_deposition_1_bc_periodic_empty_ppepc_378_stored = 0.015098538129041734
L_2_error_in_f_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_192_128_128_dts_1p83emin7_deposition_1_bc_periodic_empty_ppepc_378_stored = 0.02199724972190763
L_2_error_in_p_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_192_128_128_dts_1p83emin7_deposition_1_bc_periodic_empty_ppepc_378_stored = 0.023280860404873267
L_inf_error_in_f_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_192_128_128_dts_1p83emin7_deposition_1_bc_periodic_empty_ppepc_378_stored = 0.1355645410130646
L_inf_error_in_p_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_192_128_128_dts_1p83emin7_deposition_1_bc_periodic_empty_ppepc_378_stored = 0.06954772252985025

L_1_error_in_f_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_378_stored = 0.007516863504784114
L_1_error_in_p_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_378_stored = 0.009135536191331763
L_2_error_in_f_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_378_stored = 0.014855629734832327
L_2_error_in_p_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_378_stored = 0.01394140647178575
L_inf_error_in_f_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_378_stored = 0.13372083343563557
L_inf_error_in_p_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_378_stored = 0.04925618661745684

In [ ]:
Lx_cm = 4.212e6
ncell_x = np.array([96, 192, 384])
dx_over_R = (Lx_cm / ncell_x) / R

L_1_error_in_f_all = [
    L_1_error_in_f_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_96_64_64_dts_3p66emin7_deposition_1_bc_periodic_empty_ppepc_378_stored,
    L_1_error_in_f_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_192_128_128_dts_1p83emin7_deposition_1_bc_periodic_empty_ppepc_378_stored,
    L_1_error_in_f_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_378_stored,
]
L_1_error_in_p_all = [
    L_1_error_in_p_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_96_64_64_dts_3p66emin7_deposition_1_bc_periodic_empty_ppepc_378_stored,
    L_1_error_in_p_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_192_128_128_dts_1p83emin7_deposition_1_bc_periodic_empty_ppepc_378_stored,
    L_1_error_in_p_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_378_stored,
]
L_2_error_in_f_all = [
    L_2_error_in_f_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_96_64_64_dts_3p66emin7_deposition_1_bc_periodic_empty_ppepc_378_stored,
    L_2_error_in_f_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_192_128_128_dts_1p83emin7_deposition_1_bc_periodic_empty_ppepc_378_stored,
    L_2_error_in_f_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_378_stored,
]
L_2_error_in_p_all = [
    L_2_error_in_p_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_96_64_64_dts_3p66emin7_deposition_1_bc_periodic_empty_ppepc_378_stored,
    L_2_error_in_p_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_192_128_128_dts_1p83emin7_deposition_1_bc_periodic_empty_ppepc_378_stored,
    L_2_error_in_p_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_378_stored,
]
L_inf_error_in_f_all = [
    L_inf_error_in_f_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_96_64_64_dts_3p66emin7_deposition_1_bc_periodic_empty_ppepc_378_stored,
    L_inf_error_in_f_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_192_128_128_dts_1p83emin7_deposition_1_bc_periodic_empty_ppepc_378_stored,
    L_inf_error_in_f_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_378_stored,
]
L_inf_error_in_p_all = [
    L_inf_error_in_p_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_96_64_64_dts_3p66emin7_deposition_1_bc_periodic_empty_ppepc_378_stored,
    L_inf_error_in_p_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_192_128_128_dts_1p83emin7_deposition_1_bc_periodic_empty_ppepc_378_stored,
    L_inf_error_in_p_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_378_stored,
]
def linear_fit(x, y):
    """Least-squares fit of y = a*x + b. Returns (a, b)."""
    a, b = np.polyfit(np.asarray(x, dtype=float), np.asarray(y, dtype=float), 1)
    return float(a), float(b)

def power_law_fit(x, y):
    """Fit y = C * x^p. Returns (p, C)."""
    p, log10_C = linear_fit(np.log10(x), np.log10(y))
    return p, 10.0 ** log10_C

p_L_1_f, C_L_1_f = power_law_fit(dx_over_R, L_1_error_in_f_all)
p_L_1_p, C_L_1_p = power_law_fit(dx_over_R, L_1_error_in_p_all)
p_L_2_f, C_L_2_f = power_law_fit(dx_over_R, L_2_error_in_f_all)
p_L_2_p, C_L_2_p = power_law_fit(dx_over_R, L_2_error_in_p_all)
p_L_inf_f, C_L_inf_f = power_law_fit(dx_over_R, L_inf_error_in_f_all)
p_L_inf_p, C_L_inf_p = power_law_fit(dx_over_R, L_inf_error_in_p_all)

print(f"p_L_1_f: {p_L_1_f}")
print(f"p_L_1_p: {p_L_1_p}")
print(f"p_L_2_f: {p_L_2_f}")
print(f"p_L_2_p: {p_L_2_p}")
print(f"p_L_inf_f: {p_L_inf_f}")
print(f"p_L_inf_p: {p_L_inf_p}")

print(f"C_L_1_f: {C_L_1_f}")
print(f"C_L_1_p: {C_L_1_p}")
print(f"C_L_2_f: {C_L_2_f}")
print(f"C_L_2_p: {C_L_2_p}")
print(f"C_L_inf_f: {C_L_inf_f}")
print(f"C_L_inf_p: {C_L_inf_p}")

In [ ]:
fontsize = 28
marker_size = 12

for quantity, curves in [
    ("f", [
        (L_inf_error_in_f_all, p_L_inf_f, r"$L_{\infty}"),
        (L_1_error_in_f_all, p_L_1_f, r"$L_1"),
        (L_2_error_in_f_all, p_L_2_f, r"$L_2"),
    ]),
    ("p", [
        (L_inf_error_in_p_all, p_L_inf_p, r"$L_{\infty}"),
        (L_1_error_in_p_all, p_L_1_p, r"$L_1"),
        (L_2_error_in_p_all, p_L_2_p, r"$L_2"),
    ]),
]:
    fig, ax = plt.subplots(figsize=(10, 6))
    for errors, slope, symbol in curves:
        ax.plot(
            np.log10(dx_over_R),
            np.log10(errors),
            label=symbol + r"\propto {\Delta x_{\mathrm{cell}}} ^{" + f"{slope:.3f}" + "}$",
            linewidth=4,
            marker="o",
            markersize=marker_size,
        )
    ax.set_ylabel(r"$\log_{10}\,L_i(" + quantity + r")$", fontsize=fontsize)
    ax.set_xlabel(r"$\log_{10}\,\,\Delta x_{\mathrm{cell}} / R\,$", fontsize=fontsize)
    leg = ax.legend(framealpha=0.0, ncol=1, fontsize=fontsize)
    pf.apply_custom_settings(ax, leg, False)
    fig.tight_layout()
    plt.show()


# Convergence on npar

In [ ]:
L_1_error_in_f_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_378_stored = 0.007516863504784114
L_1_error_in_p_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_378_stored = 0.009135536191331763
L_2_error_in_f_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_378_stored = 0.014855629734832327
L_2_error_in_p_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_378_stored = 0.01394140647178575
L_inf_error_in_f_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_378_stored = 0.13372083343563557
L_inf_error_in_p_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_378_stored = 0.04925618661745684

L_1_error_in_f_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_212_stored = 0.008000744962224235
L_1_error_in_p_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_212_stored = 0.010069916173567065
L_2_error_in_f_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_212_stored = 0.01533581632490751
L_2_error_in_p_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_212_stored = 0.015529149314303757
L_inf_error_in_f_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_212_stored = 0.13145480126563802
L_inf_error_in_p_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_212_stored = 0.04621906038037066

L_1_error_in_f_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_92_stored = 0.009180256033029442
L_1_error_in_p_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_92_stored = 0.012585552149785363
L_2_error_in_f_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_92_stored = 0.018079997437701018
L_2_error_in_p_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_92_stored = 0.0197392133147952
L_inf_error_in_f_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_92_stored = 0.15843701308103553
L_inf_error_in_p_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_92_stored = 0.07102110423034658

In [ ]:
n_particles_per_cell = np.array([378, 212, 92])
Omega = 4*np.pi / n_particles_per_cell

L_1_error_in_f_all = [
    L_1_error_in_f_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_378_stored,
    L_1_error_in_f_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_212_stored,
    L_1_error_in_f_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_92_stored,
]
L_1_error_in_p_all = [
    L_1_error_in_p_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_378_stored,
    L_1_error_in_p_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_212_stored,
    L_1_error_in_p_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_92_stored,
]
L_2_error_in_f_all = [
    L_2_error_in_f_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_378_stored,
    L_2_error_in_f_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_212_stored,
    L_2_error_in_f_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_92_stored,
]
L_2_error_in_p_all = [
    L_2_error_in_p_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_378_stored,
    L_2_error_in_p_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_212_stored,
    L_2_error_in_p_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_92_stored,
]
L_inf_error_in_f_all = [
    L_inf_error_in_f_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_378_stored,
    L_inf_error_in_f_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_212_stored,
    L_inf_error_in_f_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_92_stored,
]
L_inf_error_in_p_all = [
    L_inf_error_in_p_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_378_stored,
    L_inf_error_in_p_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_212_stored,
    L_inf_error_in_p_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_92_stored,
]

def linear_fit(x, y):
    """Least-squares fit of y = a*x + b. Returns (a, b)."""
    a, b = np.polyfit(np.asarray(x, dtype=float), np.asarray(y, dtype=float), 1)
    return float(a), float(b)

def power_law_fit(x, y):
    """Fit y = C * x^p. Returns (p, C)."""
    p, log10_C = linear_fit(np.log10(x), np.log10(y))
    return p, 10.0 ** log10_C

p_L_1_f, C_L_1_f = power_law_fit(Omega, L_1_error_in_f_all)
p_L_1_p, C_L_1_p = power_law_fit(Omega, L_1_error_in_p_all)
p_L_2_f, C_L_2_f = power_law_fit(Omega, L_2_error_in_f_all)
p_L_2_p, C_L_2_p = power_law_fit(Omega, L_2_error_in_p_all)
p_L_inf_f, C_L_inf_f = power_law_fit(Omega, L_inf_error_in_f_all)
p_L_inf_p, C_L_inf_p = power_law_fit(Omega, L_inf_error_in_p_all)

print(f"p_L_1_f: {p_L_1_f}")
print(f"p_L_1_p: {p_L_1_p}")
print(f"p_L_2_f: {p_L_2_f}")
print(f"p_L_2_p: {p_L_2_p}")
print(f"p_L_inf_f: {p_L_inf_f}")
print(f"p_L_inf_p: {p_L_inf_p}")

print(f"C_L_1_f: {C_L_1_f}")
print(f"C_L_1_p: {C_L_1_p}")
print(f"C_L_2_f: {C_L_2_f}")
print(f"C_L_2_p: {C_L_2_p}")
print(f"C_L_inf_f: {C_L_inf_f}")
print(f"C_L_inf_p: {C_L_inf_p}")


In [ ]:
fontsize = 28
marker_size = 12

for quantity, curves in [
    ("f", [
        (L_inf_error_in_f_all, p_L_inf_f, r"$L_{\infty}"),
        (L_1_error_in_f_all, p_L_1_f, r"$L_1"),
        (L_2_error_in_f_all, p_L_2_f, r"$L_2"),
    ]),
    ("p", [
        (L_inf_error_in_p_all, p_L_inf_p, r"$L_{\infty}"),
        (L_1_error_in_p_all, p_L_1_p, r"$L_1"),
        (L_2_error_in_p_all, p_L_2_p, r"$L_2"),
    ]),
]:
    fig, ax = plt.subplots(figsize=(10, 6))
    for errors, slope, symbol in curves:
        ax.plot(
            np.log10(Omega),
            np.log10(errors),
            label=symbol + r"\propto \Omega^{" + f"{slope:.3f}" + "}$",
            linewidth=4,
            marker="o",
            markersize=marker_size,
        )
    ax.set_ylabel(r"$\log_{10}\,L_i(" + quantity + r")$", fontsize=fontsize)
    ax.set_xlabel(r"$\log_{10}\,\Omega\, [\mathrm{sr}]$", fontsize=fontsize)
    leg = ax.legend(framealpha=0.0, ncol=1, fontsize=fontsize)
    pf.apply_custom_settings(ax, leg, False)
    fig.tight_layout()
    plt.show()


# Plots of f and p vs r/R for every mesh resolution

In [ ]:
r_over_R_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_96_64_64_dts_3p66emin7_deposition_1_bc_periodic_empty_ppepc_378_stored = [0.0, 0.0375, 0.075, 0.1125, 0.15, 0.1875, 0.225, 0.2625, 0.3, 0.3375, 0.375, 0.4125, 0.45, 0.4875, 0.525, 0.5625, 0.6, 0.6375, 0.675, 0.7125, 0.75, 0.7875, 0.825, 0.8625, 0.9, 0.9375, 0.975, 1.0125, 1.05, 1.0875, 1.125, 1.1625, 1.2, 1.2375, 1.275, 1.3125, 1.35, 1.3875, 1.425, 1.4625, 1.5, 1.5375, 1.575, 1.6125, 1.65, 1.6875, 1.725, 1.7625, 1.8, 1.8375, 1.875, 1.9125, 1.95, 1.9875, 2.025, 2.0625, 2.1, 2.1375, 2.175, 2.2125, 2.25, 2.2875, 2.325]
f_simulation_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_96_64_64_dts_3p66emin7_deposition_1_bc_periodic_empty_ppepc_378_stored = [6.600665823677304e-16, 8.477084096720317e-06, 1.7865882578831984e-05, 2.908109870758459e-05, 4.328469949784611e-05, 6.161765220762054e-05, 8.685234239241492e-05, 0.00012232400873869075, 0.00017291924986761022, 0.0002452671966819075, 0.0003468112418698555, 0.000493106625260615, 0.0007055398780757053, 0.0010114589309385626, 0.0014610948428800026, 0.002131438673418464, 0.0031308688014512597, 0.0046070316003181625, 0.006861253871579771, 0.010304001829046306, 0.01566956930112075, 0.024190723976030844, 0.03833654788724622, 0.06310184608664353, 0.11151654113749362, 0.22928230394160098, 0.4701925579496544, 0.6728811740286734, 0.7300005708806497, 0.7653223355867573, 0.790649031908546, 0.8126235466147096, 0.8262054088907712, 0.8384016957098865, 0.8505587110619877, 0.8615552937613754, 0.8725314294161799, 0.8800111197262313, 0.8854320974311821, 0.8914226060257431, 0.8985566755951977, 0.9061552761802011, 0.9114766178751147, 0.9146460648490704, 0.9176188905336695, 0.9216466299535094, 0.9252308623596194, 0.9284594384380795, 0.9323196815693882, 0.9362237880810639, 0.9401715576747056, 0.9429746394984693, 0.9457016476878317, 0.9483011577258806, 0.9498448088341325, 0.9503106462463887, 0.9504298187110044, 0.950691333010396, 0.9517064459429635, 0.9530891982193012, 0.9546283798514023, 0.9562327855975912, 0.9579140232997982]
f_analytic_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_96_64_64_dts_3p66emin7_deposition_1_bc_periodic_empty_ppepc_378_stored = [5.55136761539776e-17, 5.781087115286238e-06, 1.2155762070824759e-05, 1.979274250018932e-05, 2.9521859537140333e-05, 4.24435110781892e-05, 6.007804870350675e-05, 8.457821796792235e-05, 0.0001190385101080526, 0.00016795222830156553, 0.00023789370902899825, 0.0003385452106251175, 0.0004842550694792665, 0.00069642213181683, 0.0010071796178843578, 0.0014651505388944656, 0.002144561770291207, 0.003159920380135072, 0.004690154878046973, 0.00701943958876066, 0.010608820493209383, 0.01622833447410945, 0.025218288658574863, 0.0400600022754023, 0.06582129135951013, 0.11479987462322089, 0.2302295568680517, 0.5845378604840445, 0.6558218335509317, 0.6990036012043547, 0.7310974128913053, 0.7566926739928154, 0.7778886646194654, 0.79586748138606, 0.8113769544347225, 0.8249269708430669, 0.8368838045266594, 0.8475208628121339, 0.8570483209834011, 0.8656315428002519, 0.8734031061620973, 0.880470970825347, 0.8869242091549948, 0.8928371378317311, 0.8982723659395674, 0.9032830885715429, 0.9079148423834069, 0.9122068695001619, 0.9161931912169518, 0.9199034631874081, 0.9233636635960156, 0.9265966523360569, 0.929622629055157, 0.9324595113985146, 0.9351232494387307, 0.9376280888539138, 0.9399867924464557, 0.9422108277069742, 0.9443105262506188, 0.946295220286128, 0.9481733596146634, 0.9499526126180692, 0.9516399535945005]
p_simulation_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_96_64_64_dts_3p66emin7_deposition_1_bc_periodic_empty_ppepc_378_stored = [0.33316510111002734, 0.33316466733218086, 0.33316339087488206, 0.33316122449118685, 0.33315796865249925, 0.33315303576733124, 0.3331457624310136, 0.3331348072436239, 0.33311847299263064, 0.33309423286999634, 0.33305915383672474, 0.3330086041947682, 0.33293533510732554, 0.3328294797804479, 0.3326730142724005, 0.3324409503964965, 0.33209820332464435, 0.3316004576191961, 0.3308666665308041, 0.3297821826377484, 0.3281618483087613, 0.3257371163127908, 0.32202309610615354, 0.3163041041088077, 0.3080787997719046, 0.3057302083351493, 0.37096840471844367, 0.4961792233032906, 0.5614603749731673, 0.6071024849603953, 0.6426048675492902, 0.6745352968659188, 0.6947262605458401, 0.7135890341041747, 0.7326117383233395, 0.7501726650956683, 0.7678755690336425, 0.7800476309252554, 0.7891658110617864, 0.7994331431713025, 0.8116560314344061, 0.8247226648595805, 0.8338773527569743, 0.8394267011661334, 0.8447246942572367, 0.8519090285946386, 0.8583255969431358, 0.8641094611308805, 0.8710781547618333, 0.8781737827441369, 0.8853474075042499, 0.8904429453177318, 0.8954363555095887, 0.9002377944417015, 0.9031071411436749, 0.9039716590898503, 0.9041924822087358, 0.9046874756727371, 0.9066064761239782, 0.9092268705217449, 0.9121487661964708, 0.9151903323339788, 0.918367072575881]
p_analytic_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_96_64_64_dts_3p66emin7_deposition_1_bc_periodic_empty_ppepc_378_stored = [0.3333335001667918, 0.3333332413841125, 0.3333324280158607, 0.33333094372124067, 0.33332857620770806, 0.3333249867512684, 0.3333196612664867, 0.33331183548344123, 0.3333003827458485, 0.3332836470270285, 0.33325919496825923, 0.3332234475681625, 0.3331711322942784, 0.3330944663300412, 0.33298193600017556, 0.33281646779098034, 0.3325726799452503, 0.3322127407478252, 0.33168011133203074, 0.3308900784552656, 0.32971545339445435, 0.3279651950762704, 0.32535366729768883, 0.32146263561146093, 0.31573048634215456, 0.3077444320033026, 0.3009870603268964, 0.39939495538372527, 0.4696505437641434, 0.518841856299449, 0.5586308174605366, 0.5923345358756703, 0.6215689696753025, 0.6473059163101776, 0.6702008870863575, 0.6907285948530941, 0.709249516522164, 0.7260466973100951, 0.741347932663717, 0.7553400412973428, 0.7681785289156751, 0.7799944014795411, 0.7908991280405997, 0.8009883535286374, 0.8103447381533713, 0.8190401696111771, 0.827137513699742, 0.8346920182642105, 0.8417524522791601, 0.8483620394259136, 0.8545592298419635, 0.8603783434645937, 0.8658501097747312, 0.8710021237476573, 0.8758592330127849, 0.880443868514405, 0.8847763281368857, 0.888875021215303, 0.8927566798390882, 0.8964365427093033, 0.8999285150672252, 0.9032453087690697, 0.9063985650312111]

r_over_R_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_192_128_128_dts_1p83emin7_deposition_1_bc_periodic_empty_ppepc_378_stored = [0.0, 0.01875, 0.0375, 0.05625, 0.075, 0.09375, 0.1125, 0.13125, 0.15, 0.16875, 0.1875, 0.20625, 0.225, 0.24375, 0.2625, 0.28125, 0.3, 0.31875, 0.3375, 0.35625, 0.375, 0.39375, 0.4125, 0.43125, 0.45, 0.46875, 0.4875, 0.50625, 0.525, 0.54375, 0.5625, 0.58125, 0.6, 0.61875, 0.6375, 0.65625, 0.675, 0.69375, 0.7125, 0.73125, 0.75, 0.76875, 0.7875, 0.80625, 0.825, 0.84375, 0.8625, 0.88125, 0.9, 0.91875, 0.9375, 0.95625, 0.975, 0.99375, 1.0125, 1.03125, 1.05, 1.06875, 1.0875, 1.10625, 1.125, 1.14375, 1.1625, 1.18125, 1.2, 1.21875, 1.2375, 1.25625, 1.275, 1.29375, 1.3125, 1.33125, 1.35, 1.36875, 1.3875, 1.40625, 1.425, 1.44375, 1.4625, 1.48125, 1.5, 1.51875, 1.5375, 1.55625, 1.575, 1.59375, 1.6125, 1.63125, 1.65, 1.66875, 1.6875, 1.70625, 1.725, 1.74375, 1.7625, 1.78125, 1.8, 1.81875, 1.8375, 1.85625, 1.875, 1.89375, 1.9125, 1.93125, 1.95, 1.96875, 1.9875, 2.00625, 2.025, 2.04375, 2.0625, 2.08125, 2.1, 2.11875, 2.1375, 2.15625, 2.175, 2.19375, 2.2125, 2.23125, 2.25, 2.26875, 2.2875, 2.30625, 2.325, 2.34375, 2.3625]
f_simulation_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_192_128_128_dts_1p83emin7_deposition_1_bc_periodic_empty_ppepc_378_stored = [-1.952123280003715e-15, 3.3836088526474385e-06, 6.8839971212701685e-06, 1.0539146779795173e-05, 1.4550874672728431e-05, 1.8932410824486217e-05, 2.3771571068620482e-05, 2.927289721565008e-05, 3.5393805582425635e-05, 4.25430900026546e-05, 5.095080415379343e-05, 6.07735720377527e-05, 7.231229383982195e-05, 8.582498552271257e-05, 0.0001017106299757764, 0.00012063460906731505, 0.00014304898632841134, 0.00017010840559078628, 0.00020220381627580604, 0.00024025688631160196, 0.00028630913716995963, 0.0003427102152108996, 0.0004109906485226942, 0.0004925621685355593, 0.0005873935025159586, 0.0007008528753073359, 0.0008384350041498969, 0.0010021098506374734, 0.001204377035069114, 0.001453864283548739, 0.001763397337666016, 0.0021407232279329383, 0.002582160515307999, 0.00311294325924065, 0.003790112556310923, 0.004624187633917755, 0.005652685292729638, 0.00693176453547531, 0.008476059195961649, 0.010407309166974604, 0.012825275035939308, 0.015818283778160108, 0.01960395238063433, 0.024482685230887598, 0.03063697537284074, 0.03841556350135346, 0.0487034714769068, 0.062202445004065035, 0.08020975726814526, 0.10543120084546752, 0.14213117560487387, 0.201720962321616, 0.31720835886097665, 0.5219411780329516, 0.6493387127484739, 0.6867452030088108, 0.7091686753684097, 0.7284113836024925, 0.74208019883444, 0.7520878717427225, 0.7640500052172468, 0.7756373786098293, 0.7876828120905517, 0.8015006038785161, 0.8091336714193368, 0.8138923427284354, 0.8190115491688769, 0.8253202212332023, 0.8344203022586373, 0.8401134043774516, 0.8442962192214646, 0.8501477326331358, 0.8579448376730876, 0.8638798310352511, 0.8687132568445229, 0.8724674261319608, 0.875399696530021, 0.8770873504494606, 0.8785941439941598, 0.881372722027244, 0.8852426143032158, 0.8905941940861682, 0.8954345912306149, 0.8992783403054596, 0.9028219769892563, 0.9061530578571814, 0.908148953605446, 0.9086202302465323, 0.9095830805987268, 0.9115114098899396, 0.9128876629123253, 0.9147011886409957, 0.9176140207753949, 0.9200617235978651, 0.9219219821646298, 0.9233191348508706, 0.9243152061355705, 0.9257808860276437, 0.927518342627769, 0.9299122244895838, 0.9325059443411303, 0.934918828064381, 0.9371259711987181, 0.9396642354270741, 0.9419574295397357, 0.9435200139395722, 0.9446392287134019, 0.9457843763001886, 0.9467903179623625, 0.9474824447209337, 0.9480457843203605, 0.9482674074558046, 0.9482146624532423, 0.9482377823100553, 0.9483815026704929, 0.9485388445614453, 0.9488545884876228, 0.9492449083005007, 0.9494561310905427, 0.9497956075817008, 0.9500127095124686, 0.9501173379746066, 0.9502369514285708, 0.9505178796171108, 0.9512890648332801, 0.9525821394184826, 0.954479138308166]
f_analytic_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_192_128_128_dts_1p83emin7_deposition_1_bc_periodic_empty_ppepc_378_stored = [5.55136761539776e-17, 2.854299149758322e-06, 5.781087115286238e-06, 8.855118816019634e-06, 1.2155762070824759e-05, 1.5769508997105106e-05, 1.979274250018932e-05, 2.4334858579141564e-05, 2.9521859537140333e-05, 3.550055232748126e-05, 4.24435110781892e-05, 5.055499448072776e-05, 6.007804870350675e-05, 7.130307672753957e-05, 8.457821796792235e-05, 0.00010032196089257351, 0.0001190385101080526, 0.00014133655321692866, 0.00016795222830156553, 0.00019977728871185515, 0.00023789370902899825, 0.0002836162890818507, 0.0003385452106251175, 0.0004046310086625377, 0.0004842550694792665, 0.0005803296044128496, 0.00069642213181683, 0.0008369109105423441, 0.0010071796178843578, 0.0012138620078409488, 0.0014651505388944656, 0.0017711873334976072, 0.002144561770291207, 0.002600947168866937, 0.003159920380135072, 0.0038460241212033832, 0.004690154878046973, 0.00573139275261159, 0.00701943958876066, 0.008617907746470895, 0.010608820493209383, 0.013098875133562787, 0.01622833447410945, 0.020183950947975655, 0.025218288658574863, 0.031679604958012766, 0.0400600022754023, 0.05107705509568965, 0.06582129135951013, 0.08604557643100985, 0.11479987462322089, 0.15806654726438138, 0.2302295568680517, 0.386376637019887, 0.5845378604840445, 0.626370888316594, 0.6558218335509317, 0.6792921792610009, 0.6990036012043547, 0.7160567286913022, 0.7310974128913053, 0.7445453504877032, 0.7566926739928154, 0.7677530421945464, 0.7778886646194654, 0.7872263303397163, 0.79586748138606, 0.8038948427041578, 0.8113769544347225, 0.8183713695167056, 0.8249269708430669, 0.8310856895267152, 0.8368838045266594, 0.8423529431397403, 0.8475208628121339, 0.8524120707553602, 0.8570483209834011, 0.861449017166822, 0.8656315428002519, 0.86961153387412, 0.8734031061620973, 0.8770190458500337, 0.880470970825347, 0.8837694677026519, 0.8869242091549948, 0.8899440546856738, 0.8928371378317311, 0.8956109419096066, 0.8982723659395674, 0.9008277825061798, 0.9032830885715429, 0.9056437502182488, 0.9079148423834069, 0.9101010840292565, 0.9122068695001619, 0.9142362965143025, 0.9161931912169518, 0.9180811307031048, 0.9199034631874081, 0.9216633263070833, 0.9233636635960156, 0.9250072393752498, 0.9265966523360569, 0.9281343478170406, 0.929622629055157, 0.9310636673412365, 0.9324595113985146, 0.9338120959337229, 0.9351232494387307, 0.9363947013930544, 0.9376280888539138, 0.9388249625475151, 0.9399867924464557, 0.9411149729719746, 0.9422108277069742, 0.9432756138350941, 0.9443105262506188, 0.945316701295922, 0.946295220286128, 0.9472471127813786, 0.9481733596146634, 0.9490748956864895, 0.9499526126180692, 0.9508073611838953, 0.9516399535945005, 0.9524511656405685, 0.9532417386560135]
p_simulation_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_192_128_128_dts_1p83emin7_deposition_1_bc_periodic_empty_ppepc_378_stored = [0.33316502900252387, 0.33316496298382053, 0.33316474630361426, 0.3331643326070224, 0.3331637293480962, 0.33316296901134024, 0.3331619531042715, 0.3331606850847139, 0.33315913237046213, 0.3331572309482579, 0.3331549106883515, 0.3331520376536287, 0.3331485270308368, 0.3331442522487973, 0.33313910488601584, 0.3331329698254866, 0.33312555380461234, 0.33311634733438517, 0.33310528266142375, 0.333092124464693, 0.3330760807609613, 0.3330562725892071, 0.3330320035232257, 0.333002892618621, 0.3329688801716808, 0.33292894001088796, 0.33288074655940697, 0.33282335377240446, 0.3327522793960251, 0.3326631865427888, 0.3325526814839734, 0.3324191569177993, 0.3322647116638109, 0.33208014330410407, 0.33185135161110507, 0.33156719297758064, 0.3312181818335681, 0.33079326998435565, 0.33028663435015343, 0.3296843429890036, 0.3289474427415212, 0.32804671872317803, 0.32694521717845215, 0.3256294765006602, 0.3240444068298849, 0.3221670821565518, 0.31996913064090937, 0.31737749019517725, 0.3143949418182133, 0.31107705963869153, 0.30757499863980203, 0.3054505149493287, 0.31478487667339666, 0.3816818007552893, 0.4680223240639753, 0.5085131716859769, 0.5354128827089939, 0.5587356776086772, 0.575683743564722, 0.5886224228411417, 0.6044718988654832, 0.6203701131390832, 0.63727093896438, 0.6568366149322791, 0.6677350885587449, 0.6749496404182251, 0.6827436338322616, 0.6923038796146568, 0.7063201141290303, 0.7152810959413687, 0.7219118189786979, 0.7311893794042441, 0.7436265954725197, 0.753115288819692, 0.760883709634563, 0.7670289528311524, 0.7718928417045967, 0.7747065977679531, 0.7772410018829379, 0.7819306641509192, 0.7884825774165402, 0.7976066823734934, 0.8058646005590882, 0.8124456486314984, 0.8185435276412847, 0.824285882378511, 0.8277664307388836, 0.8285918466684488, 0.830304266831537, 0.8337419849979628, 0.836175223159014, 0.8393733218609918, 0.8445359073940312, 0.848900043757059, 0.85222562175026, 0.8547198796073621, 0.8565069098462922, 0.8591646786352021, 0.8623158469752764, 0.8666405407822986, 0.8713304336967325, 0.8757148438450899, 0.8797304346905527, 0.8843421185119216, 0.8885093362180551, 0.8913670598895143, 0.8934182931059883, 0.8955136877021971, 0.8973556704100857, 0.8986364652122756, 0.8996834261850603, 0.9000960878310934, 0.8999925481813219, 0.9000354129770498, 0.9003066963677416, 0.9006041488720191, 0.9012027779848316, 0.9019462790179293, 0.9023501974711269, 0.9029969215370406, 0.9034039849939456, 0.903594036606518, 0.9038137132583598, 0.904343111064253, 0.9058048636710516, 0.9082500626296296, 0.9118287760558945]
p_analytic_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_192_128_128_dts_1p83emin7_deposition_1_bc_periodic_empty_ppepc_378_stored = [0.3333335001667918, 0.33333343603948484, 0.3333332413841125, 0.33333290930006804, 0.3333324280158607, 0.33333178047288414, 0.33333094372124067, 0.3333298881051677, 0.33332857620770806, 0.3333269615151708, 0.3333249867512684, 0.3333225818181617, 0.3333196612664867, 0.3333161211981414, 0.33331183548344123, 0.33330665114725966, 0.3333003827458485, 0.3332928055157656, 0.3332836470270285, 0.3332725770121266, 0.33325919496825923, 0.3332430150388642, 0.3332234475681625, 0.33319977658407374, 0.3331711322942784, 0.3331364574697344, 0.3330944663300412, 0.333043594223837, 0.33298193600017556, 0.3329071704753078, 0.33281646779098034, 0.3327063757076257, 0.3325726799452503, 0.3324102325367662, 0.3322127407478252, 0.33197250739629697, 0.33168011133203074, 0.33132401439890075, 0.3308900784552656, 0.3303609732114472, 0.32971545339445435, 0.3289274836288787, 0.3279651950762704, 0.3267896778102876, 0.32535366729768883, 0.323600321651318, 0.32146263561146093, 0.3188649457184913, 0.31573048634215456, 0.31200651018046055, 0.3077444320033026, 0.3033807214235643, 0.3009870603268964, 0.31457254857132905, 0.39939495538372527, 0.4389654491561266, 0.4696505437641434, 0.49576850304561043, 0.518841856299449, 0.5396412669346529, 0.5586308174605366, 0.5761209489339945, 0.5923345358756703, 0.6074398835464428, 0.6215689696753025, 0.6348283244106708, 0.6473059163101776, 0.6590757230032905, 0.6702008870863575, 0.6807359680431266, 0.6907285948530941, 0.7002207086259864, 0.709249516522164, 0.717848237956369, 0.7260466973100951, 0.7338718019615618, 0.741347932663717, 0.7484972655779268, 0.7553400412973428, 0.7618947910545576, 0.7681785289156751, 0.7742069158833247, 0.7799944014795411, 0.7855543461427559, 0.7908991280405997, 0.7960402363570445, 0.8009883535286374, 0.8057534279367164, 0.8103447381533713, 0.8147709503681186, 0.8190401696111771, 0.8231599855185, 0.827137513699742, 0.8309794328352347, 0.8346920182642105, 0.8382811723920387, 0.8417524522791601, 0.8451110948028077, 0.8483620394259136, 0.8515099491964553, 0.8545592298419635, 0.8575140472194928, 0.8603783434645937, 0.8631558517115508, 0.8658501097747312, 0.8684644725674188, 0.8710021237476573, 0.8734660864263312, 0.8758592330127849, 0.8781842944141817, 0.880443868514405, 0.8826404280957587, 0.8847763281368857, 0.8868538127068422, 0.888875021215303, 0.8908419943874772, 0.8927566798390882, 0.8946209371495691, 0.8964365427093033, 0.8982051942513591, 0.8999285150672252, 0.9016080579136508, 0.9032453087690697, 0.9048416902817003, 0.9063985650312111, 0.9079172386162548, 0.9093989624806526]

r_over_R_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_378_stored = [0.0, 0.009375, 0.01875, 0.028125, 0.0375, 0.046875, 0.05625, 0.065625, 0.075, 0.084375, 0.09375, 0.103125, 0.1125, 0.121875, 0.13125, 0.140625, 0.15, 0.159375, 0.16875, 0.178125, 0.1875, 0.196875, 0.20625, 0.215625, 0.225, 0.234375, 0.24375, 0.253125, 0.2625, 0.271875, 0.28125, 0.290625, 0.3, 0.309375, 0.31875, 0.328125, 0.3375, 0.346875, 0.35625, 0.365625, 0.375, 0.384375, 0.39375, 0.403125, 0.4125, 0.421875, 0.43125, 0.440625, 0.45, 0.459375, 0.46875, 0.478125, 0.4875, 0.496875, 0.50625, 0.515625, 0.525, 0.534375, 0.54375, 0.553125, 0.5625, 0.571875, 0.58125, 0.590625, 0.6, 0.609375, 0.61875, 0.628125, 0.6375, 0.646875, 0.65625, 0.665625, 0.675, 0.684375, 0.69375, 0.703125, 0.7125, 0.721875, 0.73125, 0.740625, 0.75, 0.759375, 0.76875, 0.778125, 0.7875, 0.796875, 0.80625, 0.815625, 0.825, 0.834375, 0.84375, 0.853125, 0.8625, 0.871875, 0.88125, 0.890625, 0.9, 0.909375, 0.91875, 0.928125, 0.9375, 0.946875, 0.95625, 0.965625, 0.975, 0.984375, 0.99375, 1.003125, 1.0125, 1.021875, 1.03125, 1.040625, 1.05, 1.059375, 1.06875, 1.078125, 1.0875, 1.096875, 1.10625, 1.115625, 1.125, 1.134375, 1.14375, 1.153125, 1.1625, 1.171875, 1.18125, 1.190625, 1.2, 1.209375, 1.21875, 1.228125, 1.2375, 1.246875, 1.25625, 1.265625, 1.275, 1.284375, 1.29375, 1.303125, 1.3125, 1.321875, 1.33125, 1.340625, 1.35, 1.359375, 1.36875, 1.378125, 1.3875, 1.396875, 1.40625, 1.415625, 1.425, 1.434375, 1.44375, 1.453125, 1.4625, 1.471875, 1.48125, 1.490625, 1.5, 1.509375, 1.51875, 1.528125, 1.5375, 1.546875, 1.55625, 1.565625, 1.575, 1.584375, 1.59375, 1.603125, 1.6125, 1.621875, 1.63125, 1.640625, 1.65, 1.659375, 1.66875, 1.678125, 1.6875, 1.696875, 1.70625, 1.715625, 1.725, 1.734375, 1.74375, 1.753125, 1.7625, 1.771875, 1.78125, 1.790625, 1.8, 1.809375, 1.81875, 1.828125, 1.8375, 1.846875, 1.85625, 1.865625, 1.875, 1.884375, 1.89375, 1.903125, 1.9125, 1.921875, 1.93125, 1.940625, 1.95, 1.959375, 1.96875, 1.978125, 1.9875, 1.996875, 2.00625, 2.015625, 2.025, 2.034375, 2.04375, 2.053125, 2.0625, 2.071875, 2.08125, 2.090625, 2.1, 2.109375, 2.11875, 2.128125, 2.1375, 2.146875, 2.15625, 2.165625, 2.175, 2.184375, 2.19375, 2.203125, 2.2125, 2.221875, 2.23125, 2.240625, 2.25, 2.259375, 2.26875, 2.278125, 2.2875, 2.296875, 2.30625, 2.315625, 2.325, 2.334375, 2.34375, 2.353125, 2.3625, 2.371875, 2.38125]
f_simulation_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_378_stored = [-1.4653231912650581e-15, 1.5494062712090061e-06, 3.1048815159639308e-06, 4.707517782153776e-06, 6.3194459622994995e-06, 7.941881116694654e-06, 9.646885017741837e-06, 1.1412333413484306e-05, 1.3272105328916425e-05, 1.5230844278886586e-05, 1.7286846910635118e-05, 1.9458239882573758e-05, 2.1702719872818345e-05, 2.4035744620571467e-05, 2.654637702918124e-05, 2.9263947906961522e-05, 3.226035817223958e-05, 3.542815925810375e-05, 3.876404028792895e-05, 4.24145420708325e-05, 4.6324984577540556e-05, 5.053397393164673e-05, 5.520420448912012e-05, 6.0250772553798414e-05, 6.582527608799871e-05, 7.181219932782216e-05, 7.812637913161101e-05, 8.486073119244716e-05, 9.24443296733257e-05, 0.00010086857903459902, 0.00010989664006092556, 0.00011964402277971057, 0.00013041287319977686, 0.00014226601239115467, 0.00015509788927814871, 0.00016883108012532395, 0.00018434221439033084, 0.00020081042342824903, 0.00021855000426218948, 0.00023816481296448752, 0.00025988561059054856, 0.0002840257495147693, 0.00031009944736970025, 0.00033886582557161095, 0.0003703730306614099, 0.00040497600428018757, 0.00044278185618571494, 0.00048470589839342106, 0.000530691099527211, 0.0005805811440451103, 0.0006353689788411882, 0.000695485730760384, 0.0007622386662190278, 0.0008339150320607713, 0.0009136010008492347, 0.0010044183021480828, 0.0011037045665994146, 0.0012121926827845986, 0.0013313779811108194, 0.0014633109407540593, 0.0016112019789397808, 0.0017705698109258391, 0.0019425191850462582, 0.0021333496495769366, 0.002341002650470945, 0.002573288360640905, 0.002831574962679987, 0.0031235153804175775, 0.0034509458128788957, 0.003815541410523794, 0.0042126183244966114, 0.00465571404601352, 0.005157081650135204, 0.0057019295535966534, 0.0063057930851195074, 0.00697859793216333, 0.007711258413605056, 0.008520568778394088, 0.00944862960465829, 0.01048832729495051, 0.011654803702558579, 0.012929930824261547, 0.01434738418291743, 0.015987585406032477, 0.017856709672487942, 0.019917293754929902, 0.022262298139557168, 0.0248691037064755, 0.027777892782731954, 0.031074217068047992, 0.03494926067615915, 0.03934964438866859, 0.04432728622424201, 0.05008041086870367, 0.05668301701935622, 0.06436464136591678, 0.07344217883688052, 0.08408271161821715, 0.09670344685506753, 0.11181687311356955, 0.13076505290755042, 0.15394268041087245, 0.18407450369039693, 0.22489782792308854, 0.283122492210179, 0.38045907091785297, 0.5200974704555226, 0.593657896494921, 0.6257689156460754, 0.6469965037992731, 0.6587055141151763, 0.6698812543539988, 0.6832329045398083, 0.69412891580464, 0.706297495401222, 0.7204824690570041, 0.728172682701063, 0.7333735997235629, 0.7365352700870402, 0.7420857799410969, 0.7493002457250918, 0.7567048834490675, 0.763056085695782, 0.7671249265325536, 0.7712759735389967, 0.7768691768928753, 0.7870389855433929, 0.796857669433105, 0.8018810292131779, 0.8045393672497976, 0.8057143540044845, 0.8075592695792534, 0.8099385899938065, 0.8116074744806071, 0.8131263833044853, 0.8162244392720157, 0.82334786930828, 0.8295191209945916, 0.8316076002991456, 0.8332206961838071, 0.8361859060710636, 0.8389495894283565, 0.8406240537313752, 0.8421881321743035, 0.8460548723256808, 0.8515400529365171, 0.8548407202295062, 0.85798639334674, 0.861992807352885, 0.8648974872828873, 0.8670691431294053, 0.8684402585025798, 0.8698123678455428, 0.8718552088566426, 0.8729622268828952, 0.8734869659197503, 0.8738159607624342, 0.8745808370432033, 0.8751918595989813, 0.8758208447184326, 0.8772359842367607, 0.8795702368588728, 0.8819240642098128, 0.8849982526127226, 0.8884790121912128, 0.8908059630742949, 0.8928013603752184, 0.8948341854179658, 0.8961625841466951, 0.897545828408934, 0.8994998067160117, 0.9023958103100558, 0.904664222175276, 0.9051616650062307, 0.9054272251355848, 0.9056941417590576, 0.9061302874491693, 0.9064052622214096, 0.9065141912639724, 0.9067441193879648, 0.9073180312432904, 0.908105266271359, 0.9088054498993653, 0.9100663050651627, 0.9126624088318189, 0.9153994025709509, 0.9168973988013847, 0.9170814606916439, 0.9172056258458412, 0.9176572295773445, 0.9188556459814574, 0.9203861452899103, 0.9212973470123474, 0.921583505034344, 0.9215647505503497, 0.9216546687654095, 0.9220319634101303, 0.9225441700569079, 0.9232792916869793, 0.9244125229830009, 0.926201315383525, 0.9286441527265431, 0.9305471953941057, 0.931856457857954, 0.9324727544689763, 0.9331511998750399, 0.9340919826979471, 0.9355638782454132, 0.9380929843783944, 0.9403862319260924, 0.9412856416584997, 0.9415343676927559, 0.9418042292433162, 0.9421799426110783, 0.9426541908560938, 0.9434531393642187, 0.944545285364941, 0.9456308255996226, 0.9465601532441525, 0.9469010888678754, 0.946934894838969, 0.9469137344100383, 0.9469309948358536, 0.9471236312992729, 0.947207946720965, 0.9472498117948972, 0.9472325034798257, 0.9472040809823958, 0.9472028529380401, 0.94718605773974, 0.9471592826318177, 0.9472279421211419, 0.9473308290656445, 0.947477869869787, 0.9475804568723227, 0.9476097917787889, 0.9475037946185176, 0.9473997874689809, 0.9473567457982508, 0.9474060415620005, 0.9475830671936368, 0.947886718257769, 0.94811032079269, 0.9481543195866253, 0.9481482061798522, 0.9481608065041691, 0.9483204680273846, 0.9485312085516038, 0.9487410210206745, 0.9489487696688196, 0.9491703717624562, 0.9494862865794155, 0.9497995469893135, 0.9503494691274467, 0.9512135706781771]
f_analytic_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_378_stored = [5.55136761539776e-17, 1.4226453844488892e-06, 2.854299149758322e-06, 4.3040398677015174e-06, 5.781087115286238e-06, 7.294873539634997e-06, 8.855118816019634e-06, 1.0471906158919991e-05, 1.2155762070824759e-05, 1.3917740047695134e-05, 1.5769508997105106e-05, 1.7723447175192477e-05, 1.979274250018932e-05, 2.1991500170138952e-05, 2.4334858579141564e-05, 2.683911461817557e-05, 2.9521859537140333e-05, 3.2402126654953777e-05, 3.550055232748126e-05, 3.883955171910129e-05, 4.24435110781892e-05, 4.633899839095476e-05, 5.055499448072776e-05, 5.5123146837194206e-05, 6.007804870350675e-05, 6.545754621979247e-05, 7.130307672753957e-05, 7.766004167775974e-05, 8.457821796792235e-05, 9.211221195757216e-05, 0.00010032196089257351, 0.00010927328699863067, 0.0001190385101080526, 0.0001296971253534113, 0.00014133655321692866, 0.00015405297007176002, 0.00016795222830156553, 0.0001831508761571072, 0.00019977728871185515, 0.00021797292263442769, 0.00023789370902899825, 0.0002597116003161396, 0.0002836162890818507, 0.0003098171190239678, 0.0003385452106251175, 0.00037005582701069104, 0.0004046310086625377, 0.00044258250931061567, 0.0004842550694792665, 0.0005300300689007807, 0.0005803296044128496, 0.0006356210461374085, 0.00069642213181683, 0.0007633066673045405, 0.0008369109105423441, 0.000917940727114868, 0.0010071796178843578, 0.0011054977335754898, 0.0012138620078409488, 0.0013333475597085895, 0.0014651505388944656, 0.0016106026138662558, 0.0017711873334976072, 0.0019485586295632463, 0.002144561770291207, 0.0023612571260683774, 0.002600947168866937, 0.002866207199110918, 0.003159920380135072, 0.003485317764386654, 0.0038460241212033832, 0.004246110528633489, 0.004690154878046973, 0.005183311668848142, 0.00573139275261159, 0.006340961035976801, 0.00701943958876066, 0.00777523915327358, 0.008617907746470895, 0.009558306933720359, 0.010608820493209383, 0.011783602667815068, 0.013098875133562787, 0.014573284364412607, 0.01622833447410945, 0.01808891520024694, 0.020183950947975655, 0.02254720544603093, 0.025218288658574863, 0.028243929779589675, 0.031679604958012766, 0.035591644903829996, 0.0400600022754023, 0.045181942690287885, 0.05107705509568965, 0.05789419027645254, 0.06582129135951013, 0.07509969366982727, 0.08604557643100985, 0.0990833396806949, 0.11479987462322089, 0.1340377365488795, 0.15806654726438138, 0.18892846420318454, 0.2302295568680517, 0.2893480965757367, 0.386376637019887, 0.5493835073960827, 0.5845378604840445, 0.6078416621161116, 0.626370888316594, 0.6420689210111367, 0.6558218335509317, 0.668125536139601, 0.6792921792610009, 0.6895340315554566, 0.6990036012043547, 0.7078150733665282, 0.7160567286913022, 0.7237985973154231, 0.7310974128913053, 0.7379999503928002, 0.7445453504877032, 0.7507667841402815, 0.7566926739928154, 0.7623476090131328, 0.7677530421945464, 0.7729278314578334, 0.7778886646194654, 0.7826503982762003, 0.7872263303397163, 0.7916284222552058, 0.79586748138606, 0.7999533121767038, 0.8038948427041578, 0.8077002308216117, 0.8113769544347225, 0.8149318882213095, 0.8183713695167056, 0.8217012552149952, 0.8249269708430669, 0.8280535535549015, 0.8310856895267152, 0.8340277468339151, 0.8368838045266594, 0.8396576782967587, 0.8423529431397403, 0.8449729538118492, 0.8475208628121339, 0.8499996367154934, 0.8524120707553602, 0.8547608019494454, 0.8570483209834011, 0.8592769829456988, 0.861449017166822, 0.8635665359561419, 0.8656315428002519, 0.8676459396902758, 0.86961153387412, 0.8715300440929749, 0.8734031061620973, 0.8752322782855431, 0.8770190458500337, 0.8787648258045205, 0.880470970825347, 0.882138773064978, 0.8837694677026519, 0.8853642361684176, 0.8869242091549948, 0.8884504694918003, 0.8899440546856738, 0.8914059594203257, 0.8928371378317311, 0.8942385055781367, 0.8956109419096066, 0.8969552914313924, 0.8982723659395674, 0.8995629460041125, 0.9008277825061798, 0.9020675981159711, 0.9032830885715429, 0.9044749240277881, 0.9056437502182488, 0.9067901895928038, 0.9079148423834069, 0.909018287609802, 0.9101010840292565, 0.911163771024533, 0.9122068695001619, 0.9132308825903072, 0.9142362965143025, 0.9152235812448821, 0.9161931912169518, 0.9171455659616641, 0.9180811307031048, 0.9190002969753333, 0.9199034631874081, 0.9207910150776087, 0.9216633263070833, 0.9225207588738824, 0.9233636635960156, 0.9241923805143687, 0.9250072393752498, 0.925808559916222, 0.9265966523360569, 0.9273718176071983, 0.9281343478170406, 0.9288845265413425, 0.929622629055157, 0.9303489227524407, 0.9310636673412365, 0.9317671151476852, 0.9324595113985146, 0.9331410944260201, 0.9338120959337229, 0.9344727412284164, 0.9351232494387307, 0.9357638336960733, 0.9363947013930544, 0.9370160543020428, 0.9376280888539138, 0.9382309962092267, 0.9388249625475151, 0.9394101691104987, 0.9399867924464557, 0.9405550045576676, 0.9411149729719746, 0.9416668609797317, 0.9422108277069742, 0.9427470282536156, 0.9432756138350941, 0.9437967319114089, 0.9443105262506188, 0.9448171371141588, 0.945316701295922, 0.9458093522683008, 0.946295220286128, 0.9467744324312597, 0.9472471127813786, 0.9477133824315959, 0.9481733596146634, 0.94862715978414, 0.9490748956864895, 0.9495166774416431, 0.9499526126180692, 0.950382806299971, 0.9508073611838953, 0.9512263775880764, 0.9516399535945005, 0.9520481850426744, 0.9524511656405685, 0.9528489869836524, 0.9532417386560135, 0.9536295082434366, 0.9540123814145287]
p_simulation_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_378_stored = [0.33316502318785435, 0.33316500803255866, 0.33316494774881533, 0.33316486161320763, 0.3331647363986787, 0.3331645795136557, 0.33316439156687727, 0.3331641451380356, 0.3331638473783065, 0.3331635120650711, 0.3331631149940581, 0.33316266123515414, 0.333162180980435, 0.3331616734685603, 0.3331610972647536, 0.33316044197278843, 0.3331596773783648, 0.33315883676128627, 0.33315792735899524, 0.33315693768965576, 0.3331558352626884, 0.33315460074104275, 0.3331532176421054, 0.33315166113271466, 0.33314990623229546, 0.3331480021906574, 0.3331460061455418, 0.33314388245124127, 0.33314142630911503, 0.33313866678564996, 0.3331356615532933, 0.3331324007707045, 0.3331287246763851, 0.3331246762561338, 0.3331203112722692, 0.3331156520840429, 0.33311039643079987, 0.3331047409784025, 0.33309860510896044, 0.3330918586520036, 0.3330842982350518, 0.3330758615421645, 0.3330666616650786, 0.33305647067328265, 0.3330453114857397, 0.33303296003247096, 0.3330193474491096, 0.33300420109323897, 0.3329877636071154, 0.3329700342221912, 0.33295045223759595, 0.33292865943888733, 0.3329050387224929, 0.33287920143406946, 0.3328506598624101, 0.33281826233180767, 0.3327823907491678, 0.33274292759303026, 0.33269990310430775, 0.3326521422037821, 0.33259910689520433, 0.3325420493587358, 0.3324810795867951, 0.33241429904133046, 0.33234183577822574, 0.33226088728733766, 0.3321705800173269, 0.33206842489500293, 0.3319535090889914, 0.33182646869230553, 0.3316873277625835, 0.3315329056388638, 0.3313627342256758, 0.3311749624149387, 0.3309729933565857, 0.3307504396583754, 0.33050802727232836, 0.3302405806387753, 0.32994349681752794, 0.3296133706952643, 0.32925035474808595, 0.32884971478860586, 0.32841076940482017, 0.3279141812567609, 0.3273590896518938, 0.3267353083062687, 0.32607152185125193, 0.325347926728136, 0.3245601840419619, 0.3236638093794643, 0.3226825061448326, 0.32158623834256583, 0.3203791171025886, 0.3190810293912356, 0.3175994785378992, 0.31601701178976765, 0.31430631461263087, 0.3123700389083138, 0.31029405324660514, 0.3080386070468666, 0.3058345977104352, 0.30335562368391855, 0.3013781082252848, 0.3004310011816503, 0.3021216930815784, 0.31578769782307986, 0.3638287351887859, 0.41149599746636795, 0.441393702112316, 0.4622605454334761, 0.47486594294063333, 0.4873325905351397, 0.5026168665635875, 0.5153621298516382, 0.5297420543221084, 0.5467203579795086, 0.5563393816119185, 0.5628913892594632, 0.5668490460970363, 0.5742246803216283, 0.5838004451467068, 0.593583754773627, 0.6022134934222686, 0.6077945568759304, 0.6134821389327589, 0.6213467172772932, 0.635748488684355, 0.6495887878590677, 0.6567262051430494, 0.6605486631911163, 0.6622869340017613, 0.6650497502207179, 0.6685787290208309, 0.671087377790143, 0.6734268584538409, 0.6781410919580755, 0.6889947439104618, 0.6984196525017375, 0.7016048476367026, 0.7040544325277565, 0.7086875862413246, 0.7130658005006211, 0.71573598650818, 0.7182129389295302, 0.7243172884141138, 0.7330029970730211, 0.738267567756753, 0.7433000876539848, 0.7497214184748818, 0.7544000929371708, 0.7579338415210087, 0.760178375805231, 0.7624629683815856, 0.765832537136988, 0.7676514086887637, 0.7685344376208713, 0.7690899840036539, 0.7703800963515084, 0.7714245780827286, 0.7724776402564161, 0.7748388755162855, 0.7787677526863209, 0.7827562287377777, 0.7879610317012089, 0.7938358292972726, 0.797787492147566, 0.801199874496898, 0.8046834604286203, 0.8069521625119863, 0.809323862173147, 0.8126621965660359, 0.8176280510758023, 0.8215228414088847, 0.8223736070094915, 0.8228534876695679, 0.823338077753864, 0.8241160357330565, 0.8246037724674109, 0.824799414073919, 0.8252192345557355, 0.8262375284312244, 0.8276317493373124, 0.8288654954561768, 0.8310827918890982, 0.835669870597586, 0.8405165521631285, 0.8431666202300919, 0.8434867330369513, 0.843699711623461, 0.8444991464532088, 0.8466298117061868, 0.8493558329439311, 0.8509894025553422, 0.851511108135782, 0.8514697524681372, 0.851624386213443, 0.8523082703266792, 0.853245041703044, 0.854574089366505, 0.8566234220849724, 0.8598569601470627, 0.8642713692207182, 0.8677074337705873, 0.8700753486742432, 0.8711942123810991, 0.8724293415881161, 0.8741414215482813, 0.876810415463513, 0.8813986065158971, 0.8855679472838415, 0.8872108952872303, 0.8876665290427685, 0.888161562285583, 0.8888519576703516, 0.8897245347462727, 0.8911864837083547, 0.8931873833844742, 0.8951759890569977, 0.8968798367797135, 0.8975093362485019, 0.8975710936829704, 0.8975307205667441, 0.897562470212421, 0.8979264416282147, 0.8980850483233137, 0.8981646039430193, 0.8981339885130734, 0.8980824459820772, 0.8980812029581258, 0.8980491355641608, 0.8979988804002452, 0.8981276910545894, 0.8983205942783129, 0.8985974971565517, 0.8987905437196207, 0.8988461883475988, 0.8986447369745213, 0.8984467665017913, 0.8983651601549496, 0.8984602795688618, 0.8987964799913096, 0.8993706915248523, 0.899792354586698, 0.899873290672065, 0.8998605545131276, 0.8998841119376622, 0.900188177462267, 0.9005888631169898, 0.9009854280312103, 0.9013760821702455, 0.90179404239275, 0.9023915634551587, 0.9029819853456074, 0.9040173503650105, 0.9056438774286545]
p_analytic_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_378_stored = [0.3333335001667918, 0.3333334841703318, 0.33333343603948484, 0.33333335534860326, 0.3333332413841125, 0.33333309313822623, 0.33333290930006804, 0.3333326882441153, 0.3333324280158607, 0.3333321263145568, 0.33333178047288414, 0.3333313874333565, 0.33333094372124067, 0.33333044541374207, 0.3333298881051677, 0.3333292668677418, 0.33332857620770806, 0.33332781001630735, 0.3333269615151708, 0.33332602319561544, 0.3333249867512684, 0.3333238430033868, 0.3333225818181617, 0.33332119201522514, 0.3333196612664867, 0.333317975984336, 0.3333161211981414, 0.3333140804178589, 0.33331183548344123, 0.333309366398592, 0.33330665114725966, 0.3333036654910869, 0.3333003827458485, 0.33329677353469145, 0.3332928055157656, 0.333288443081566, 0.3332836470270285, 0.33327837418309636, 0.3332725770121266, 0.33326620316111183, 0.33325919496825923, 0.3332514889179866, 0.3332430150388642, 0.3332336962384312, 0.3332234475681625, 0.33321217541112846, 0.33319977658407374, 0.33318613734474345, 0.3331711322942784, 0.33315462316338285, 0.3331364574697344, 0.3331164670327183, 0.3330944663300412, 0.33307025067906915, 0.333043594223837, 0.33301424770656607, 0.33298193600017556, 0.33294635537566175, 0.3329071704753078, 0.33286401095945695, 0.33281646779098034, 0.3327640891175808, 0.3327063757076257, 0.33264277589028035, 0.3325726799452503, 0.3324954138813901, 0.3324102325367662, 0.3323163119253966, 0.3322127407478252, 0.3320985109738665, 0.33197250739629697, 0.3318334960439837, 0.33168011133203074, 0.33151084181514673, 0.33132401439890075, 0.3311177768523294, 0.3308900784552656, 0.3306386486060123, 0.3303609732114472, 0.3300542686851901, 0.32971545339445435, 0.3293411164292555, 0.3289274836288787, 0.328470380905382, 0.3279651950762704, 0.3274068326953552, 0.3267896778102876, 0.32610755026943633, 0.32535366729768883, 0.3245206128017896, 0.323600321651318, 0.32258409068795535, 0.32146263561146093, 0.32022622524182415, 0.3188649457184913, 0.31736918400953823, 0.31573048634215456, 0.31394307050739073, 0.31200651018046055, 0.30993059631729836, 0.3077444320033026, 0.30551425813066635, 0.3033807214235643, 0.30164414940832995, 0.3009870603268964, 0.30318928773938136, 0.31457254857132905, 0.3698513106744584, 0.39939495538372527, 0.4208545258781095, 0.4389654491561266, 0.45503170614317257, 0.4696505437641434, 0.4831583368260397, 0.49576850304561043, 0.5076275446236284, 0.518841856299449, 0.5294920519966053, 0.5396412669346529, 0.5493402775280881, 0.5586308174605366, 0.5675478143042283, 0.5761209489339945, 0.5843757739352092, 0.5923345358756703, 0.600016792294252, 0.6074398835464428, 0.614619299730829, 0.6215689696753025, 0.6283014925276198, 0.6348283244106708, 0.6411599315381966, 0.6473059163101776, 0.6532751223322006, 0.6590757230032905, 0.6647152959689813, 0.6702008870863575, 0.6755390649401664, 0.6807359680431266, 0.685797345960717, 0.6907285948530941, 0.6955347890597163, 0.7002207086259864, 0.704790863727414, 0.709249516522164, 0.7136007005587626, 0.717848237956369, 0.7219957552674428, 0.7260466973100951, 0.730004340021069, 0.7338718019615618, 0.7376520547578475, 0.741347932663717, 0.7449621412593401, 0.7484972655779268, 0.7519557772224539, 0.7553400412973428, 0.7586523225385777, 0.7618947910545576, 0.7650695277167726, 0.7681785289156751, 0.771223711277841, 0.7742069158833247, 0.7771299121204943, 0.7799944014795411, 0.7828020209188232, 0.7855543461427559, 0.7882528945525505, 0.7908991280405997, 0.7934944557365813, 0.7960402363570445, 0.7985377806381738, 0.8009883535286374, 0.8033931761603241, 0.8057534279367164, 0.8080702482052372, 0.8103447381533713, 0.8125779623627047, 0.8147709503681186, 0.8169246981929251, 0.8190401696111771, 0.8211182975982944, 0.8231599855185, 0.8251661083270179, 0.827137513699742, 0.8290750231070635, 0.8309794328352347, 0.8328515149416306, 0.8346920182642105, 0.8365016691357117, 0.8382811723920387, 0.8400312120821907, 0.8417524522791601, 0.8434455377862861, 0.8451110948028077, 0.8467497316497141, 0.8483620394259136, 0.8499485924876754, 0.8515099491964553, 0.8530466523646855, 0.8545592298419635, 0.8560481949739599, 0.8575140472194928, 0.8589572724285777, 0.8603783434645937, 0.8617777205560136, 0.8631558517115508, 0.8645131732046425, 0.8658501097747312, 0.8671670752156528, 0.8684644725674188, 0.869742694513506, 0.8710021237476573, 0.8722431332076148, 0.8734660864263312, 0.8746713378291187, 0.8758592330127849, 0.8770301089621981, 0.8781842944141817, 0.879322109971411, 0.880443868514405, 0.8815498752386376, 0.8826404280957587, 0.8837158177882123, 0.8847763281368857, 0.8858222362751498, 0.8868538127068422, 0.8878713216721088, 0.888875021215303, 0.8898651633756595, 0.8908419943874772, 0.8918057548600413, 0.8927566798390882, 0.8936949990960839, 0.8946209371495691, 0.8955347134867095, 0.8964365427093033, 0.897326634571719, 0.8982051942513591, 0.8990724223478707, 0.8999285150672252, 0.9007736643388232, 0.9016080579136508, 0.9024318794797638, 0.9032453087690697, 0.9040485216508699, 0.9048416902817003, 0.9056249830935535, 0.9063985650312111, 0.9071625975137676, 0.9079172386162548, 0.9086626430804998, 0.9093989624806526, 0.9101263452248605, 0.9108449366853398]


In [ ]:
runs = [
    (96, "radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_96_64_64_dts_3p66emin7_deposition_1_bc_periodic_empty_ppepc_378_stored"),
    (192, "radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_192_128_128_dts_1p83emin7_deposition_1_bc_periodic_empty_ppepc_378_stored"),
    (384, "radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_378_stored"),
]

def stored(name, suffix):
    return np.asarray(globals()[f"{name}_{suffix}"], dtype=float)

for quantity, sim_name, analytic_name in [
    ("f", "f_simulation", "f_analytic"),
    ("p", "p_simulation", "p_analytic"),
]:
    fig, ax = plt.subplots(figsize=(10, 6))
    for color_index, (nx, suffix) in enumerate(runs):
        ax.plot(
            stored("r_over_R", suffix),
            stored(sim_name, suffix),
            color=f"C{color_index}",
            linestyle="dotted",
            linewidth=4,
            label=rf"$\hat{{\mathrm{{{quantity}}}}}_{{\mathrm{{sim}}}},\, N_x = {nx}$",
        )
    r_fine = stored("r_over_R", runs[-1][1])
    ax.plot(
        r_fine,
        stored(analytic_name, runs[-1][1]),
        color="k",
        linestyle="solid",
        alpha=0.5,
        linewidth=4,
        label=rf"$\hat{{\mathrm{{{quantity}}}}}_{{\mathrm{{analytic}}}}$",
    )
    # ax.axvline(1.0, color="k", linestyle=":", linewidth=1.5, label=r"$r = R$")
    ax.set_xlabel(r"$r / R$")
    ax.set_ylabel(rf"${quantity}$")
    ax.set_xlim(0.0, None)
    leg = ax.legend(framealpha=0.0, ncol=1, fontsize=16)
    pf.apply_custom_settings(ax, leg, False)
    fig.tight_layout()
    plt.show()


# Plots of f and p vs r/R for every angular resolution

In [ ]:
r_over_R_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_378_stored = [0.0, 0.009375, 0.01875, 0.028125, 0.0375, 0.046875, 0.05625, 0.065625, 0.075, 0.084375, 0.09375, 0.103125, 0.1125, 0.121875, 0.13125, 0.140625, 0.15, 0.159375, 0.16875, 0.178125, 0.1875, 0.196875, 0.20625, 0.215625, 0.225, 0.234375, 0.24375, 0.253125, 0.2625, 0.271875, 0.28125, 0.290625, 0.3, 0.309375, 0.31875, 0.328125, 0.3375, 0.346875, 0.35625, 0.365625, 0.375, 0.384375, 0.39375, 0.403125, 0.4125, 0.421875, 0.43125, 0.440625, 0.45, 0.459375, 0.46875, 0.478125, 0.4875, 0.496875, 0.50625, 0.515625, 0.525, 0.534375, 0.54375, 0.553125, 0.5625, 0.571875, 0.58125, 0.590625, 0.6, 0.609375, 0.61875, 0.628125, 0.6375, 0.646875, 0.65625, 0.665625, 0.675, 0.684375, 0.69375, 0.703125, 0.7125, 0.721875, 0.73125, 0.740625, 0.75, 0.759375, 0.76875, 0.778125, 0.7875, 0.796875, 0.80625, 0.815625, 0.825, 0.834375, 0.84375, 0.853125, 0.8625, 0.871875, 0.88125, 0.890625, 0.9, 0.909375, 0.91875, 0.928125, 0.9375, 0.946875, 0.95625, 0.965625, 0.975, 0.984375, 0.99375, 1.003125, 1.0125, 1.021875, 1.03125, 1.040625, 1.05, 1.059375, 1.06875, 1.078125, 1.0875, 1.096875, 1.10625, 1.115625, 1.125, 1.134375, 1.14375, 1.153125, 1.1625, 1.171875, 1.18125, 1.190625, 1.2, 1.209375, 1.21875, 1.228125, 1.2375, 1.246875, 1.25625, 1.265625, 1.275, 1.284375, 1.29375, 1.303125, 1.3125, 1.321875, 1.33125, 1.340625, 1.35, 1.359375, 1.36875, 1.378125, 1.3875, 1.396875, 1.40625, 1.415625, 1.425, 1.434375, 1.44375, 1.453125, 1.4625, 1.471875, 1.48125, 1.490625, 1.5, 1.509375, 1.51875, 1.528125, 1.5375, 1.546875, 1.55625, 1.565625, 1.575, 1.584375, 1.59375, 1.603125, 1.6125, 1.621875, 1.63125, 1.640625, 1.65, 1.659375, 1.66875, 1.678125, 1.6875, 1.696875, 1.70625, 1.715625, 1.725, 1.734375, 1.74375, 1.753125, 1.7625, 1.771875, 1.78125, 1.790625, 1.8, 1.809375, 1.81875, 1.828125, 1.8375, 1.846875, 1.85625, 1.865625, 1.875, 1.884375, 1.89375, 1.903125, 1.9125, 1.921875, 1.93125, 1.940625, 1.95, 1.959375, 1.96875, 1.978125, 1.9875, 1.996875, 2.00625, 2.015625, 2.025, 2.034375, 2.04375, 2.053125, 2.0625, 2.071875, 2.08125, 2.090625, 2.1, 2.109375, 2.11875, 2.128125, 2.1375, 2.146875, 2.15625, 2.165625, 2.175, 2.184375, 2.19375, 2.203125, 2.2125, 2.221875, 2.23125, 2.240625, 2.25, 2.259375, 2.26875, 2.278125, 2.2875, 2.296875, 2.30625, 2.315625, 2.325, 2.334375, 2.34375, 2.353125, 2.3625, 2.371875, 2.38125]
f_simulation_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_378_stored = [-1.4653231912650581e-15, 1.5494062712090061e-06, 3.1048815159639308e-06, 4.707517782153776e-06, 6.3194459622994995e-06, 7.941881116694654e-06, 9.646885017741837e-06, 1.1412333413484306e-05, 1.3272105328916425e-05, 1.5230844278886586e-05, 1.7286846910635118e-05, 1.9458239882573758e-05, 2.1702719872818345e-05, 2.4035744620571467e-05, 2.654637702918124e-05, 2.9263947906961522e-05, 3.226035817223958e-05, 3.542815925810375e-05, 3.876404028792895e-05, 4.24145420708325e-05, 4.6324984577540556e-05, 5.053397393164673e-05, 5.520420448912012e-05, 6.0250772553798414e-05, 6.582527608799871e-05, 7.181219932782216e-05, 7.812637913161101e-05, 8.486073119244716e-05, 9.24443296733257e-05, 0.00010086857903459902, 0.00010989664006092556, 0.00011964402277971057, 0.00013041287319977686, 0.00014226601239115467, 0.00015509788927814871, 0.00016883108012532395, 0.00018434221439033084, 0.00020081042342824903, 0.00021855000426218948, 0.00023816481296448752, 0.00025988561059054856, 0.0002840257495147693, 0.00031009944736970025, 0.00033886582557161095, 0.0003703730306614099, 0.00040497600428018757, 0.00044278185618571494, 0.00048470589839342106, 0.000530691099527211, 0.0005805811440451103, 0.0006353689788411882, 0.000695485730760384, 0.0007622386662190278, 0.0008339150320607713, 0.0009136010008492347, 0.0010044183021480828, 0.0011037045665994146, 0.0012121926827845986, 0.0013313779811108194, 0.0014633109407540593, 0.0016112019789397808, 0.0017705698109258391, 0.0019425191850462582, 0.0021333496495769366, 0.002341002650470945, 0.002573288360640905, 0.002831574962679987, 0.0031235153804175775, 0.0034509458128788957, 0.003815541410523794, 0.0042126183244966114, 0.00465571404601352, 0.005157081650135204, 0.0057019295535966534, 0.0063057930851195074, 0.00697859793216333, 0.007711258413605056, 0.008520568778394088, 0.00944862960465829, 0.01048832729495051, 0.011654803702558579, 0.012929930824261547, 0.01434738418291743, 0.015987585406032477, 0.017856709672487942, 0.019917293754929902, 0.022262298139557168, 0.0248691037064755, 0.027777892782731954, 0.031074217068047992, 0.03494926067615915, 0.03934964438866859, 0.04432728622424201, 0.05008041086870367, 0.05668301701935622, 0.06436464136591678, 0.07344217883688052, 0.08408271161821715, 0.09670344685506753, 0.11181687311356955, 0.13076505290755042, 0.15394268041087245, 0.18407450369039693, 0.22489782792308854, 0.283122492210179, 0.38045907091785297, 0.5200974704555226, 0.593657896494921, 0.6257689156460754, 0.6469965037992731, 0.6587055141151763, 0.6698812543539988, 0.6832329045398083, 0.69412891580464, 0.706297495401222, 0.7204824690570041, 0.728172682701063, 0.7333735997235629, 0.7365352700870402, 0.7420857799410969, 0.7493002457250918, 0.7567048834490675, 0.763056085695782, 0.7671249265325536, 0.7712759735389967, 0.7768691768928753, 0.7870389855433929, 0.796857669433105, 0.8018810292131779, 0.8045393672497976, 0.8057143540044845, 0.8075592695792534, 0.8099385899938065, 0.8116074744806071, 0.8131263833044853, 0.8162244392720157, 0.82334786930828, 0.8295191209945916, 0.8316076002991456, 0.8332206961838071, 0.8361859060710636, 0.8389495894283565, 0.8406240537313752, 0.8421881321743035, 0.8460548723256808, 0.8515400529365171, 0.8548407202295062, 0.85798639334674, 0.861992807352885, 0.8648974872828873, 0.8670691431294053, 0.8684402585025798, 0.8698123678455428, 0.8718552088566426, 0.8729622268828952, 0.8734869659197503, 0.8738159607624342, 0.8745808370432033, 0.8751918595989813, 0.8758208447184326, 0.8772359842367607, 0.8795702368588728, 0.8819240642098128, 0.8849982526127226, 0.8884790121912128, 0.8908059630742949, 0.8928013603752184, 0.8948341854179658, 0.8961625841466951, 0.897545828408934, 0.8994998067160117, 0.9023958103100558, 0.904664222175276, 0.9051616650062307, 0.9054272251355848, 0.9056941417590576, 0.9061302874491693, 0.9064052622214096, 0.9065141912639724, 0.9067441193879648, 0.9073180312432904, 0.908105266271359, 0.9088054498993653, 0.9100663050651627, 0.9126624088318189, 0.9153994025709509, 0.9168973988013847, 0.9170814606916439, 0.9172056258458412, 0.9176572295773445, 0.9188556459814574, 0.9203861452899103, 0.9212973470123474, 0.921583505034344, 0.9215647505503497, 0.9216546687654095, 0.9220319634101303, 0.9225441700569079, 0.9232792916869793, 0.9244125229830009, 0.926201315383525, 0.9286441527265431, 0.9305471953941057, 0.931856457857954, 0.9324727544689763, 0.9331511998750399, 0.9340919826979471, 0.9355638782454132, 0.9380929843783944, 0.9403862319260924, 0.9412856416584997, 0.9415343676927559, 0.9418042292433162, 0.9421799426110783, 0.9426541908560938, 0.9434531393642187, 0.944545285364941, 0.9456308255996226, 0.9465601532441525, 0.9469010888678754, 0.946934894838969, 0.9469137344100383, 0.9469309948358536, 0.9471236312992729, 0.947207946720965, 0.9472498117948972, 0.9472325034798257, 0.9472040809823958, 0.9472028529380401, 0.94718605773974, 0.9471592826318177, 0.9472279421211419, 0.9473308290656445, 0.947477869869787, 0.9475804568723227, 0.9476097917787889, 0.9475037946185176, 0.9473997874689809, 0.9473567457982508, 0.9474060415620005, 0.9475830671936368, 0.947886718257769, 0.94811032079269, 0.9481543195866253, 0.9481482061798522, 0.9481608065041691, 0.9483204680273846, 0.9485312085516038, 0.9487410210206745, 0.9489487696688196, 0.9491703717624562, 0.9494862865794155, 0.9497995469893135, 0.9503494691274467, 0.9512135706781771]
f_analytic_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_378_stored = [5.55136761539776e-17, 1.4226453844488892e-06, 2.854299149758322e-06, 4.3040398677015174e-06, 5.781087115286238e-06, 7.294873539634997e-06, 8.855118816019634e-06, 1.0471906158919991e-05, 1.2155762070824759e-05, 1.3917740047695134e-05, 1.5769508997105106e-05, 1.7723447175192477e-05, 1.979274250018932e-05, 2.1991500170138952e-05, 2.4334858579141564e-05, 2.683911461817557e-05, 2.9521859537140333e-05, 3.2402126654953777e-05, 3.550055232748126e-05, 3.883955171910129e-05, 4.24435110781892e-05, 4.633899839095476e-05, 5.055499448072776e-05, 5.5123146837194206e-05, 6.007804870350675e-05, 6.545754621979247e-05, 7.130307672753957e-05, 7.766004167775974e-05, 8.457821796792235e-05, 9.211221195757216e-05, 0.00010032196089257351, 0.00010927328699863067, 0.0001190385101080526, 0.0001296971253534113, 0.00014133655321692866, 0.00015405297007176002, 0.00016795222830156553, 0.0001831508761571072, 0.00019977728871185515, 0.00021797292263442769, 0.00023789370902899825, 0.0002597116003161396, 0.0002836162890818507, 0.0003098171190239678, 0.0003385452106251175, 0.00037005582701069104, 0.0004046310086625377, 0.00044258250931061567, 0.0004842550694792665, 0.0005300300689007807, 0.0005803296044128496, 0.0006356210461374085, 0.00069642213181683, 0.0007633066673045405, 0.0008369109105423441, 0.000917940727114868, 0.0010071796178843578, 0.0011054977335754898, 0.0012138620078409488, 0.0013333475597085895, 0.0014651505388944656, 0.0016106026138662558, 0.0017711873334976072, 0.0019485586295632463, 0.002144561770291207, 0.0023612571260683774, 0.002600947168866937, 0.002866207199110918, 0.003159920380135072, 0.003485317764386654, 0.0038460241212033832, 0.004246110528633489, 0.004690154878046973, 0.005183311668848142, 0.00573139275261159, 0.006340961035976801, 0.00701943958876066, 0.00777523915327358, 0.008617907746470895, 0.009558306933720359, 0.010608820493209383, 0.011783602667815068, 0.013098875133562787, 0.014573284364412607, 0.01622833447410945, 0.01808891520024694, 0.020183950947975655, 0.02254720544603093, 0.025218288658574863, 0.028243929779589675, 0.031679604958012766, 0.035591644903829996, 0.0400600022754023, 0.045181942690287885, 0.05107705509568965, 0.05789419027645254, 0.06582129135951013, 0.07509969366982727, 0.08604557643100985, 0.0990833396806949, 0.11479987462322089, 0.1340377365488795, 0.15806654726438138, 0.18892846420318454, 0.2302295568680517, 0.2893480965757367, 0.386376637019887, 0.5493835073960827, 0.5845378604840445, 0.6078416621161116, 0.626370888316594, 0.6420689210111367, 0.6558218335509317, 0.668125536139601, 0.6792921792610009, 0.6895340315554566, 0.6990036012043547, 0.7078150733665282, 0.7160567286913022, 0.7237985973154231, 0.7310974128913053, 0.7379999503928002, 0.7445453504877032, 0.7507667841402815, 0.7566926739928154, 0.7623476090131328, 0.7677530421945464, 0.7729278314578334, 0.7778886646194654, 0.7826503982762003, 0.7872263303397163, 0.7916284222552058, 0.79586748138606, 0.7999533121767038, 0.8038948427041578, 0.8077002308216117, 0.8113769544347225, 0.8149318882213095, 0.8183713695167056, 0.8217012552149952, 0.8249269708430669, 0.8280535535549015, 0.8310856895267152, 0.8340277468339151, 0.8368838045266594, 0.8396576782967587, 0.8423529431397403, 0.8449729538118492, 0.8475208628121339, 0.8499996367154934, 0.8524120707553602, 0.8547608019494454, 0.8570483209834011, 0.8592769829456988, 0.861449017166822, 0.8635665359561419, 0.8656315428002519, 0.8676459396902758, 0.86961153387412, 0.8715300440929749, 0.8734031061620973, 0.8752322782855431, 0.8770190458500337, 0.8787648258045205, 0.880470970825347, 0.882138773064978, 0.8837694677026519, 0.8853642361684176, 0.8869242091549948, 0.8884504694918003, 0.8899440546856738, 0.8914059594203257, 0.8928371378317311, 0.8942385055781367, 0.8956109419096066, 0.8969552914313924, 0.8982723659395674, 0.8995629460041125, 0.9008277825061798, 0.9020675981159711, 0.9032830885715429, 0.9044749240277881, 0.9056437502182488, 0.9067901895928038, 0.9079148423834069, 0.909018287609802, 0.9101010840292565, 0.911163771024533, 0.9122068695001619, 0.9132308825903072, 0.9142362965143025, 0.9152235812448821, 0.9161931912169518, 0.9171455659616641, 0.9180811307031048, 0.9190002969753333, 0.9199034631874081, 0.9207910150776087, 0.9216633263070833, 0.9225207588738824, 0.9233636635960156, 0.9241923805143687, 0.9250072393752498, 0.925808559916222, 0.9265966523360569, 0.9273718176071983, 0.9281343478170406, 0.9288845265413425, 0.929622629055157, 0.9303489227524407, 0.9310636673412365, 0.9317671151476852, 0.9324595113985146, 0.9331410944260201, 0.9338120959337229, 0.9344727412284164, 0.9351232494387307, 0.9357638336960733, 0.9363947013930544, 0.9370160543020428, 0.9376280888539138, 0.9382309962092267, 0.9388249625475151, 0.9394101691104987, 0.9399867924464557, 0.9405550045576676, 0.9411149729719746, 0.9416668609797317, 0.9422108277069742, 0.9427470282536156, 0.9432756138350941, 0.9437967319114089, 0.9443105262506188, 0.9448171371141588, 0.945316701295922, 0.9458093522683008, 0.946295220286128, 0.9467744324312597, 0.9472471127813786, 0.9477133824315959, 0.9481733596146634, 0.94862715978414, 0.9490748956864895, 0.9495166774416431, 0.9499526126180692, 0.950382806299971, 0.9508073611838953, 0.9512263775880764, 0.9516399535945005, 0.9520481850426744, 0.9524511656405685, 0.9528489869836524, 0.9532417386560135, 0.9536295082434366, 0.9540123814145287]
p_simulation_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_378_stored = [0.33316502318785435, 0.33316500803255866, 0.33316494774881533, 0.33316486161320763, 0.3331647363986787, 0.3331645795136557, 0.33316439156687727, 0.3331641451380356, 0.3331638473783065, 0.3331635120650711, 0.3331631149940581, 0.33316266123515414, 0.333162180980435, 0.3331616734685603, 0.3331610972647536, 0.33316044197278843, 0.3331596773783648, 0.33315883676128627, 0.33315792735899524, 0.33315693768965576, 0.3331558352626884, 0.33315460074104275, 0.3331532176421054, 0.33315166113271466, 0.33314990623229546, 0.3331480021906574, 0.3331460061455418, 0.33314388245124127, 0.33314142630911503, 0.33313866678564996, 0.3331356615532933, 0.3331324007707045, 0.3331287246763851, 0.3331246762561338, 0.3331203112722692, 0.3331156520840429, 0.33311039643079987, 0.3331047409784025, 0.33309860510896044, 0.3330918586520036, 0.3330842982350518, 0.3330758615421645, 0.3330666616650786, 0.33305647067328265, 0.3330453114857397, 0.33303296003247096, 0.3330193474491096, 0.33300420109323897, 0.3329877636071154, 0.3329700342221912, 0.33295045223759595, 0.33292865943888733, 0.3329050387224929, 0.33287920143406946, 0.3328506598624101, 0.33281826233180767, 0.3327823907491678, 0.33274292759303026, 0.33269990310430775, 0.3326521422037821, 0.33259910689520433, 0.3325420493587358, 0.3324810795867951, 0.33241429904133046, 0.33234183577822574, 0.33226088728733766, 0.3321705800173269, 0.33206842489500293, 0.3319535090889914, 0.33182646869230553, 0.3316873277625835, 0.3315329056388638, 0.3313627342256758, 0.3311749624149387, 0.3309729933565857, 0.3307504396583754, 0.33050802727232836, 0.3302405806387753, 0.32994349681752794, 0.3296133706952643, 0.32925035474808595, 0.32884971478860586, 0.32841076940482017, 0.3279141812567609, 0.3273590896518938, 0.3267353083062687, 0.32607152185125193, 0.325347926728136, 0.3245601840419619, 0.3236638093794643, 0.3226825061448326, 0.32158623834256583, 0.3203791171025886, 0.3190810293912356, 0.3175994785378992, 0.31601701178976765, 0.31430631461263087, 0.3123700389083138, 0.31029405324660514, 0.3080386070468666, 0.3058345977104352, 0.30335562368391855, 0.3013781082252848, 0.3004310011816503, 0.3021216930815784, 0.31578769782307986, 0.3638287351887859, 0.41149599746636795, 0.441393702112316, 0.4622605454334761, 0.47486594294063333, 0.4873325905351397, 0.5026168665635875, 0.5153621298516382, 0.5297420543221084, 0.5467203579795086, 0.5563393816119185, 0.5628913892594632, 0.5668490460970363, 0.5742246803216283, 0.5838004451467068, 0.593583754773627, 0.6022134934222686, 0.6077945568759304, 0.6134821389327589, 0.6213467172772932, 0.635748488684355, 0.6495887878590677, 0.6567262051430494, 0.6605486631911163, 0.6622869340017613, 0.6650497502207179, 0.6685787290208309, 0.671087377790143, 0.6734268584538409, 0.6781410919580755, 0.6889947439104618, 0.6984196525017375, 0.7016048476367026, 0.7040544325277565, 0.7086875862413246, 0.7130658005006211, 0.71573598650818, 0.7182129389295302, 0.7243172884141138, 0.7330029970730211, 0.738267567756753, 0.7433000876539848, 0.7497214184748818, 0.7544000929371708, 0.7579338415210087, 0.760178375805231, 0.7624629683815856, 0.765832537136988, 0.7676514086887637, 0.7685344376208713, 0.7690899840036539, 0.7703800963515084, 0.7714245780827286, 0.7724776402564161, 0.7748388755162855, 0.7787677526863209, 0.7827562287377777, 0.7879610317012089, 0.7938358292972726, 0.797787492147566, 0.801199874496898, 0.8046834604286203, 0.8069521625119863, 0.809323862173147, 0.8126621965660359, 0.8176280510758023, 0.8215228414088847, 0.8223736070094915, 0.8228534876695679, 0.823338077753864, 0.8241160357330565, 0.8246037724674109, 0.824799414073919, 0.8252192345557355, 0.8262375284312244, 0.8276317493373124, 0.8288654954561768, 0.8310827918890982, 0.835669870597586, 0.8405165521631285, 0.8431666202300919, 0.8434867330369513, 0.843699711623461, 0.8444991464532088, 0.8466298117061868, 0.8493558329439311, 0.8509894025553422, 0.851511108135782, 0.8514697524681372, 0.851624386213443, 0.8523082703266792, 0.853245041703044, 0.854574089366505, 0.8566234220849724, 0.8598569601470627, 0.8642713692207182, 0.8677074337705873, 0.8700753486742432, 0.8711942123810991, 0.8724293415881161, 0.8741414215482813, 0.876810415463513, 0.8813986065158971, 0.8855679472838415, 0.8872108952872303, 0.8876665290427685, 0.888161562285583, 0.8888519576703516, 0.8897245347462727, 0.8911864837083547, 0.8931873833844742, 0.8951759890569977, 0.8968798367797135, 0.8975093362485019, 0.8975710936829704, 0.8975307205667441, 0.897562470212421, 0.8979264416282147, 0.8980850483233137, 0.8981646039430193, 0.8981339885130734, 0.8980824459820772, 0.8980812029581258, 0.8980491355641608, 0.8979988804002452, 0.8981276910545894, 0.8983205942783129, 0.8985974971565517, 0.8987905437196207, 0.8988461883475988, 0.8986447369745213, 0.8984467665017913, 0.8983651601549496, 0.8984602795688618, 0.8987964799913096, 0.8993706915248523, 0.899792354586698, 0.899873290672065, 0.8998605545131276, 0.8998841119376622, 0.900188177462267, 0.9005888631169898, 0.9009854280312103, 0.9013760821702455, 0.90179404239275, 0.9023915634551587, 0.9029819853456074, 0.9040173503650105, 0.9056438774286545]
p_analytic_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_378_stored = [0.3333335001667918, 0.3333334841703318, 0.33333343603948484, 0.33333335534860326, 0.3333332413841125, 0.33333309313822623, 0.33333290930006804, 0.3333326882441153, 0.3333324280158607, 0.3333321263145568, 0.33333178047288414, 0.3333313874333565, 0.33333094372124067, 0.33333044541374207, 0.3333298881051677, 0.3333292668677418, 0.33332857620770806, 0.33332781001630735, 0.3333269615151708, 0.33332602319561544, 0.3333249867512684, 0.3333238430033868, 0.3333225818181617, 0.33332119201522514, 0.3333196612664867, 0.333317975984336, 0.3333161211981414, 0.3333140804178589, 0.33331183548344123, 0.333309366398592, 0.33330665114725966, 0.3333036654910869, 0.3333003827458485, 0.33329677353469145, 0.3332928055157656, 0.333288443081566, 0.3332836470270285, 0.33327837418309636, 0.3332725770121266, 0.33326620316111183, 0.33325919496825923, 0.3332514889179866, 0.3332430150388642, 0.3332336962384312, 0.3332234475681625, 0.33321217541112846, 0.33319977658407374, 0.33318613734474345, 0.3331711322942784, 0.33315462316338285, 0.3331364574697344, 0.3331164670327183, 0.3330944663300412, 0.33307025067906915, 0.333043594223837, 0.33301424770656607, 0.33298193600017556, 0.33294635537566175, 0.3329071704753078, 0.33286401095945695, 0.33281646779098034, 0.3327640891175808, 0.3327063757076257, 0.33264277589028035, 0.3325726799452503, 0.3324954138813901, 0.3324102325367662, 0.3323163119253966, 0.3322127407478252, 0.3320985109738665, 0.33197250739629697, 0.3318334960439837, 0.33168011133203074, 0.33151084181514673, 0.33132401439890075, 0.3311177768523294, 0.3308900784552656, 0.3306386486060123, 0.3303609732114472, 0.3300542686851901, 0.32971545339445435, 0.3293411164292555, 0.3289274836288787, 0.328470380905382, 0.3279651950762704, 0.3274068326953552, 0.3267896778102876, 0.32610755026943633, 0.32535366729768883, 0.3245206128017896, 0.323600321651318, 0.32258409068795535, 0.32146263561146093, 0.32022622524182415, 0.3188649457184913, 0.31736918400953823, 0.31573048634215456, 0.31394307050739073, 0.31200651018046055, 0.30993059631729836, 0.3077444320033026, 0.30551425813066635, 0.3033807214235643, 0.30164414940832995, 0.3009870603268964, 0.30318928773938136, 0.31457254857132905, 0.3698513106744584, 0.39939495538372527, 0.4208545258781095, 0.4389654491561266, 0.45503170614317257, 0.4696505437641434, 0.4831583368260397, 0.49576850304561043, 0.5076275446236284, 0.518841856299449, 0.5294920519966053, 0.5396412669346529, 0.5493402775280881, 0.5586308174605366, 0.5675478143042283, 0.5761209489339945, 0.5843757739352092, 0.5923345358756703, 0.600016792294252, 0.6074398835464428, 0.614619299730829, 0.6215689696753025, 0.6283014925276198, 0.6348283244106708, 0.6411599315381966, 0.6473059163101776, 0.6532751223322006, 0.6590757230032905, 0.6647152959689813, 0.6702008870863575, 0.6755390649401664, 0.6807359680431266, 0.685797345960717, 0.6907285948530941, 0.6955347890597163, 0.7002207086259864, 0.704790863727414, 0.709249516522164, 0.7136007005587626, 0.717848237956369, 0.7219957552674428, 0.7260466973100951, 0.730004340021069, 0.7338718019615618, 0.7376520547578475, 0.741347932663717, 0.7449621412593401, 0.7484972655779268, 0.7519557772224539, 0.7553400412973428, 0.7586523225385777, 0.7618947910545576, 0.7650695277167726, 0.7681785289156751, 0.771223711277841, 0.7742069158833247, 0.7771299121204943, 0.7799944014795411, 0.7828020209188232, 0.7855543461427559, 0.7882528945525505, 0.7908991280405997, 0.7934944557365813, 0.7960402363570445, 0.7985377806381738, 0.8009883535286374, 0.8033931761603241, 0.8057534279367164, 0.8080702482052372, 0.8103447381533713, 0.8125779623627047, 0.8147709503681186, 0.8169246981929251, 0.8190401696111771, 0.8211182975982944, 0.8231599855185, 0.8251661083270179, 0.827137513699742, 0.8290750231070635, 0.8309794328352347, 0.8328515149416306, 0.8346920182642105, 0.8365016691357117, 0.8382811723920387, 0.8400312120821907, 0.8417524522791601, 0.8434455377862861, 0.8451110948028077, 0.8467497316497141, 0.8483620394259136, 0.8499485924876754, 0.8515099491964553, 0.8530466523646855, 0.8545592298419635, 0.8560481949739599, 0.8575140472194928, 0.8589572724285777, 0.8603783434645937, 0.8617777205560136, 0.8631558517115508, 0.8645131732046425, 0.8658501097747312, 0.8671670752156528, 0.8684644725674188, 0.869742694513506, 0.8710021237476573, 0.8722431332076148, 0.8734660864263312, 0.8746713378291187, 0.8758592330127849, 0.8770301089621981, 0.8781842944141817, 0.879322109971411, 0.880443868514405, 0.8815498752386376, 0.8826404280957587, 0.8837158177882123, 0.8847763281368857, 0.8858222362751498, 0.8868538127068422, 0.8878713216721088, 0.888875021215303, 0.8898651633756595, 0.8908419943874772, 0.8918057548600413, 0.8927566798390882, 0.8936949990960839, 0.8946209371495691, 0.8955347134867095, 0.8964365427093033, 0.897326634571719, 0.8982051942513591, 0.8990724223478707, 0.8999285150672252, 0.9007736643388232, 0.9016080579136508, 0.9024318794797638, 0.9032453087690697, 0.9040485216508699, 0.9048416902817003, 0.9056249830935535, 0.9063985650312111, 0.9071625975137676, 0.9079172386162548, 0.9086626430804998, 0.9093989624806526, 0.9101263452248605, 0.9108449366853398]

r_over_R_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_212_stored = [0.0, 0.009375, 0.01875, 0.028125, 0.0375, 0.046875, 0.05625, 0.065625, 0.075, 0.084375, 0.09375, 0.103125, 0.1125, 0.121875, 0.13125, 0.140625, 0.15, 0.159375, 0.16875, 0.178125, 0.1875, 0.196875, 0.20625, 0.215625, 0.225, 0.234375, 0.24375, 0.253125, 0.2625, 0.271875, 0.28125, 0.290625, 0.3, 0.309375, 0.31875, 0.328125, 0.3375, 0.346875, 0.35625, 0.365625, 0.375, 0.384375, 0.39375, 0.403125, 0.4125, 0.421875, 0.43125, 0.440625, 0.45, 0.459375, 0.46875, 0.478125, 0.4875, 0.496875, 0.50625, 0.515625, 0.525, 0.534375, 0.54375, 0.553125, 0.5625, 0.571875, 0.58125, 0.590625, 0.6, 0.609375, 0.61875, 0.628125, 0.6375, 0.646875, 0.65625, 0.665625, 0.675, 0.684375, 0.69375, 0.703125, 0.7125, 0.721875, 0.73125, 0.740625, 0.75, 0.759375, 0.76875, 0.778125, 0.7875, 0.796875, 0.80625, 0.815625, 0.825, 0.834375, 0.84375, 0.853125, 0.8625, 0.871875, 0.88125, 0.890625, 0.9, 0.909375, 0.91875, 0.928125, 0.9375, 0.946875, 0.95625, 0.965625, 0.975, 0.984375, 0.99375, 1.003125, 1.0125, 1.021875, 1.03125, 1.040625, 1.05, 1.059375, 1.06875, 1.078125, 1.0875, 1.096875, 1.10625, 1.115625, 1.125, 1.134375, 1.14375, 1.153125, 1.1625, 1.171875, 1.18125, 1.190625, 1.2, 1.209375, 1.21875, 1.228125, 1.2375, 1.246875, 1.25625, 1.265625, 1.275, 1.284375, 1.29375, 1.303125, 1.3125, 1.321875, 1.33125, 1.340625, 1.35, 1.359375, 1.36875, 1.378125, 1.3875, 1.396875, 1.40625, 1.415625, 1.425, 1.434375, 1.44375, 1.453125, 1.4625, 1.471875, 1.48125, 1.490625, 1.5, 1.509375, 1.51875, 1.528125, 1.5375, 1.546875, 1.55625, 1.565625, 1.575, 1.584375, 1.59375, 1.603125, 1.6125, 1.621875, 1.63125, 1.640625, 1.65, 1.659375, 1.66875, 1.678125, 1.6875, 1.696875, 1.70625, 1.715625, 1.725, 1.734375, 1.74375, 1.753125, 1.7625, 1.771875, 1.78125, 1.790625, 1.8, 1.809375, 1.81875, 1.828125, 1.8375, 1.846875, 1.85625, 1.865625, 1.875, 1.884375, 1.89375, 1.903125, 1.9125, 1.921875, 1.93125, 1.940625, 1.95, 1.959375, 1.96875, 1.978125, 1.9875, 1.996875, 2.00625, 2.015625, 2.025, 2.034375, 2.04375, 2.053125, 2.0625, 2.071875, 2.08125, 2.090625, 2.1, 2.109375, 2.11875, 2.128125, 2.1375, 2.146875, 2.15625, 2.165625, 2.175, 2.184375, 2.19375, 2.203125, 2.2125, 2.221875, 2.23125, 2.240625, 2.25, 2.259375, 2.26875, 2.278125, 2.2875, 2.296875, 2.30625, 2.315625, 2.325, 2.334375, 2.34375, 2.353125, 2.3625, 2.371875, 2.38125]
f_simulation_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_212_stored = [-8.079080513884522e-15, 1.5197015914449532e-06, 3.099690664516431e-06, 4.703908499629253e-06, 6.3274473656502125e-06, 7.98844184860138e-06, 9.682770007177303e-06, 1.1454726716626918e-05, 1.3323064225998236e-05, 1.525179505204797e-05, 1.7317818898925816e-05, 1.9466481285139844e-05, 2.170964625202422e-05, 2.411904581849975e-05, 2.6705533968495867e-05, 2.952405638825606e-05, 3.2485020428391066e-05, 3.5642656821560515e-05, 3.902630587502077e-05, 4.262185305495765e-05, 4.6573618506312846e-05, 5.078056629777452e-05, 5.529096663285642e-05, 6.0303989068983977e-05, 6.584883111903397e-05, 7.182510603337332e-05, 7.818273293008627e-05, 8.500986800345752e-05, 9.265512391050558e-05, 0.00010103280080487778, 0.00011014010174385049, 0.00011980063025030745, 0.0001304337478988587, 0.00014218888653977653, 0.00015512231607564983, 0.0001692576373969039, 0.0001843635715342564, 0.00020084264124226836, 0.00021893692137758594, 0.00023832354260670745, 0.0002595648530323712, 0.0002833326105032651, 0.00030933849352099424, 0.0003378346002917468, 0.00036966759255872763, 0.00040516083571222843, 0.00044368841980058657, 0.00048539938898010216, 0.000531791130941281, 0.0005814998914448778, 0.000635319243190062, 0.000697264720702583, 0.0007670949210351692, 0.0008420742284926538, 0.000925340102348003, 0.001014690397497028, 0.0011097500340586889, 0.001212182955254038, 0.0013284463423087192, 0.0014626507821948378, 0.0016114749025069185, 0.0017738434799722245, 0.0019482317244783633, 0.002141025828934656, 0.0023533819123623284, 0.0025840415448319092, 0.0028449681924396855, 0.0031311534696573055, 0.0034534468953181688, 0.0038128521601843066, 0.004205821535933944, 0.00464413871104727, 0.005136538702939775, 0.005684861410897151, 0.006287517441821731, 0.0069529898436662425, 0.007692570261348961, 0.008510391500990566, 0.009428066376614051, 0.010491302581354698, 0.01171756785394224, 0.013064654137578336, 0.014521230236995976, 0.016098509697232834, 0.017870304723063536, 0.019920653815788995, 0.02220994035877156, 0.02476920392731563, 0.02773238895479611, 0.031224174967777896, 0.0351573744799672, 0.0395847147763162, 0.04450032479884055, 0.050124177483998764, 0.05673128172341573, 0.06459754326557046, 0.07372816166707537, 0.0843388019365766, 0.09676144125479623, 0.11220493764192965, 0.13130678314039654, 0.15446032393164302, 0.1846043105146147, 0.22559896431091028, 0.2843411648137965, 0.38091612610614756, 0.517831438285525, 0.5976063458927985, 0.6298168115307421, 0.6445288453199742, 0.6543552550965244, 0.6665008585787999, 0.6827982627226677, 0.6962119955737847, 0.7086719125524856, 0.7160856557669119, 0.7237792122612948, 0.7351104758526132, 0.7439428201488549, 0.747230062091036, 0.7496750761962306, 0.7537929072535796, 0.7611565499894537, 0.767124857285259, 0.7712567955026258, 0.775128509504093, 0.7776992466545253, 0.7822623633767658, 0.7899554407919741, 0.7956789041486629, 0.8003427128438904, 0.8084026410006055, 0.815269709908036, 0.8214684520244864, 0.8273739659058728, 0.8309597425579185, 0.8317505036501452, 0.8326088756756914, 0.8333532718236094, 0.8341425540550311, 0.8356440283725229, 0.8381831151147155, 0.841227905358161, 0.8432867814581849, 0.8457732969573304, 0.8490507428545176, 0.8508933808953029, 0.8515597524429194, 0.8526626175618066, 0.8549409641585856, 0.8578150870185801, 0.8595508588075658, 0.8603820189125623, 0.8612904028671675, 0.8624825465157386, 0.8644706445468829, 0.8667301571399666, 0.8693241969103714, 0.8717235914477294, 0.873792880871827, 0.8764188062878795, 0.882929717592266, 0.8913470881782074, 0.896015617645741, 0.8989013048554267, 0.9021635133495401, 0.9048173539083612, 0.9063338486841179, 0.907577186006361, 0.9085805538873523, 0.9090301748770772, 0.9091108484177512, 0.9090860714610416, 0.9091009782433352, 0.9091981227890712, 0.9093067557978392, 0.9094915679029179, 0.9098017810508986, 0.9100422753428953, 0.9101677370446054, 0.9102075753953156, 0.91026746341124, 0.9103256581675333, 0.9105185364049221, 0.910664712217611, 0.9108376995398936, 0.9111076479496419, 0.9113899940889049, 0.911973964323846, 0.9130818333545373, 0.914548629024012, 0.916348182163887, 0.9179747776158402, 0.9184286623515812, 0.9184726964264531, 0.9187418527952316, 0.9191235877801646, 0.919343891160616, 0.9194520176220838, 0.9194468036236922, 0.9197080771662045, 0.9200467233137445, 0.9202897890160712, 0.9206606721659499, 0.9210488705879135, 0.9216359678579948, 0.9225061510853972, 0.9242435100636778, 0.9275688311263757, 0.932025220921201, 0.936068450162774, 0.9391578885201177, 0.9418994408144294, 0.9430001501721237, 0.9431255853847494, 0.9432440219217958, 0.9435417966056735, 0.9438758662659071, 0.9439063005707279, 0.9437394920583001, 0.9435794360846806, 0.9435824138829468, 0.9439346825653877, 0.9442408849952872, 0.9444607935503897, 0.9446881494459994, 0.9448219120548034, 0.9449172231831054, 0.9450691300489008, 0.9451954994119744, 0.9453683851554264, 0.9457257748692269, 0.9463277022918525, 0.9471743979671101, 0.9483281751040591, 0.9503360391418606, 0.9527745960462158, 0.9541903084155988, 0.9549808057725497, 0.9562554436727773, 0.9580415061916108, 0.96009617691206, 0.9625237764499259, 0.9641767161964461, 0.9646362175962163, 0.9647168048934353, 0.9647164926732227, 0.9646992541126815, 0.9647039168831504, 0.964703128332878, 0.9647062123424442, 0.9647289889218185, 0.9647510521532593, 0.964768133109109, 0.9647867096515732]
f_analytic_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_212_stored = [5.55136761539776e-17, 1.4226453844488892e-06, 2.854299149758322e-06, 4.3040398677015174e-06, 5.781087115286238e-06, 7.294873539634997e-06, 8.855118816019634e-06, 1.0471906158919991e-05, 1.2155762070824759e-05, 1.3917740047695134e-05, 1.5769508997105106e-05, 1.7723447175192477e-05, 1.979274250018932e-05, 2.1991500170138952e-05, 2.4334858579141564e-05, 2.683911461817557e-05, 2.9521859537140333e-05, 3.2402126654953777e-05, 3.550055232748126e-05, 3.883955171910129e-05, 4.24435110781892e-05, 4.633899839095476e-05, 5.055499448072776e-05, 5.5123146837194206e-05, 6.007804870350675e-05, 6.545754621979247e-05, 7.130307672753957e-05, 7.766004167775974e-05, 8.457821796792235e-05, 9.211221195757216e-05, 0.00010032196089257351, 0.00010927328699863067, 0.0001190385101080526, 0.0001296971253534113, 0.00014133655321692866, 0.00015405297007176002, 0.00016795222830156553, 0.0001831508761571072, 0.00019977728871185515, 0.00021797292263442769, 0.00023789370902899825, 0.0002597116003161396, 0.0002836162890818507, 0.0003098171190239678, 0.0003385452106251175, 0.00037005582701069104, 0.0004046310086625377, 0.00044258250931061567, 0.0004842550694792665, 0.0005300300689007807, 0.0005803296044128496, 0.0006356210461374085, 0.00069642213181683, 0.0007633066673045405, 0.0008369109105423441, 0.000917940727114868, 0.0010071796178843578, 0.0011054977335754898, 0.0012138620078409488, 0.0013333475597085895, 0.0014651505388944656, 0.0016106026138662558, 0.0017711873334976072, 0.0019485586295632463, 0.002144561770291207, 0.0023612571260683774, 0.002600947168866937, 0.002866207199110918, 0.003159920380135072, 0.003485317764386654, 0.0038460241212033832, 0.004246110528633489, 0.004690154878046973, 0.005183311668848142, 0.00573139275261159, 0.006340961035976801, 0.00701943958876066, 0.00777523915327358, 0.008617907746470895, 0.009558306933720359, 0.010608820493209383, 0.011783602667815068, 0.013098875133562787, 0.014573284364412607, 0.01622833447410945, 0.01808891520024694, 0.020183950947975655, 0.02254720544603093, 0.025218288658574863, 0.028243929779589675, 0.031679604958012766, 0.035591644903829996, 0.0400600022754023, 0.045181942690287885, 0.05107705509568965, 0.05789419027645254, 0.06582129135951013, 0.07509969366982727, 0.08604557643100985, 0.0990833396806949, 0.11479987462322089, 0.1340377365488795, 0.15806654726438138, 0.18892846420318454, 0.2302295568680517, 0.2893480965757367, 0.386376637019887, 0.5493835073960827, 0.5845378604840445, 0.6078416621161116, 0.626370888316594, 0.6420689210111367, 0.6558218335509317, 0.668125536139601, 0.6792921792610009, 0.6895340315554566, 0.6990036012043547, 0.7078150733665282, 0.7160567286913022, 0.7237985973154231, 0.7310974128913053, 0.7379999503928002, 0.7445453504877032, 0.7507667841402815, 0.7566926739928154, 0.7623476090131328, 0.7677530421945464, 0.7729278314578334, 0.7778886646194654, 0.7826503982762003, 0.7872263303397163, 0.7916284222552058, 0.79586748138606, 0.7999533121767038, 0.8038948427041578, 0.8077002308216117, 0.8113769544347225, 0.8149318882213095, 0.8183713695167056, 0.8217012552149952, 0.8249269708430669, 0.8280535535549015, 0.8310856895267152, 0.8340277468339151, 0.8368838045266594, 0.8396576782967587, 0.8423529431397403, 0.8449729538118492, 0.8475208628121339, 0.8499996367154934, 0.8524120707553602, 0.8547608019494454, 0.8570483209834011, 0.8592769829456988, 0.861449017166822, 0.8635665359561419, 0.8656315428002519, 0.8676459396902758, 0.86961153387412, 0.8715300440929749, 0.8734031061620973, 0.8752322782855431, 0.8770190458500337, 0.8787648258045205, 0.880470970825347, 0.882138773064978, 0.8837694677026519, 0.8853642361684176, 0.8869242091549948, 0.8884504694918003, 0.8899440546856738, 0.8914059594203257, 0.8928371378317311, 0.8942385055781367, 0.8956109419096066, 0.8969552914313924, 0.8982723659395674, 0.8995629460041125, 0.9008277825061798, 0.9020675981159711, 0.9032830885715429, 0.9044749240277881, 0.9056437502182488, 0.9067901895928038, 0.9079148423834069, 0.909018287609802, 0.9101010840292565, 0.911163771024533, 0.9122068695001619, 0.9132308825903072, 0.9142362965143025, 0.9152235812448821, 0.9161931912169518, 0.9171455659616641, 0.9180811307031048, 0.9190002969753333, 0.9199034631874081, 0.9207910150776087, 0.9216633263070833, 0.9225207588738824, 0.9233636635960156, 0.9241923805143687, 0.9250072393752498, 0.925808559916222, 0.9265966523360569, 0.9273718176071983, 0.9281343478170406, 0.9288845265413425, 0.929622629055157, 0.9303489227524407, 0.9310636673412365, 0.9317671151476852, 0.9324595113985146, 0.9331410944260201, 0.9338120959337229, 0.9344727412284164, 0.9351232494387307, 0.9357638336960733, 0.9363947013930544, 0.9370160543020428, 0.9376280888539138, 0.9382309962092267, 0.9388249625475151, 0.9394101691104987, 0.9399867924464557, 0.9405550045576676, 0.9411149729719746, 0.9416668609797317, 0.9422108277069742, 0.9427470282536156, 0.9432756138350941, 0.9437967319114089, 0.9443105262506188, 0.9448171371141588, 0.945316701295922, 0.9458093522683008, 0.946295220286128, 0.9467744324312597, 0.9472471127813786, 0.9477133824315959, 0.9481733596146634, 0.94862715978414, 0.9490748956864895, 0.9495166774416431, 0.9499526126180692, 0.950382806299971, 0.9508073611838953, 0.9512263775880764, 0.9516399535945005, 0.9520481850426744, 0.9524511656405685, 0.9528489869836524, 0.9532417386560135, 0.9536295082434366, 0.9540123814145287]
p_simulation_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_212_stored = [0.33399282525598983, 0.33399281959962024, 0.33399278156660633, 0.33399270441255974, 0.33399259910420914, 0.33399244087953295, 0.33399224319494825, 0.33399201378955273, 0.3339917260332905, 0.33399138592346067, 0.3339909927902017, 0.33399058153375416, 0.3339901088520068, 0.33398955952716103, 0.3339889257812809, 0.3339881984161663, 0.3339874356242755, 0.33398663309365617, 0.33398575520089546, 0.3339847829601061, 0.3339836732227599, 0.33398245699184004, 0.3339810929525913, 0.33397959153015583, 0.33397791202828264, 0.33397604514237883, 0.33397404047246976, 0.3339718457287957, 0.33396941785352563, 0.3339666526190575, 0.3339636429552557, 0.3339604863037128, 0.3339569262178624, 0.33395296544252584, 0.33394862779894974, 0.3339438407502919, 0.33393866482475243, 0.3339330185339635, 0.3339268533370299, 0.3339201548935417, 0.33391283408053457, 0.3339047195536097, 0.3338957001835465, 0.33388569366887666, 0.3338744394670233, 0.3338616570296017, 0.3338478402866318, 0.3338327788058484, 0.3338161916694527, 0.3337985389342698, 0.33377946725160385, 0.3337572741261707, 0.33373182252193745, 0.3337043442508342, 0.3336741905778551, 0.3336419107282759, 0.333608819945331, 0.33357315272554655, 0.3335318087123933, 0.3334838372983988, 0.3334303495957631, 0.33337243191828486, 0.3333101147755147, 0.3332424723391872, 0.3331694571072857, 0.3330886881475799, 0.33299876955992824, 0.33289945903059726, 0.33278877296190024, 0.3326654794952538, 0.33253008284256413, 0.3323790773191031, 0.3322103412638672, 0.3320264836418339, 0.33182861527534047, 0.3316108400695123, 0.33137069743522996, 0.3311037780705349, 0.3308021994925379, 0.330457798173259, 0.3300712007441865, 0.32964764475267627, 0.3291999182981551, 0.32872900582009174, 0.3282015009422425, 0.32761587238471956, 0.32697313344260737, 0.32625904358911395, 0.32546491765177576, 0.3245617690392165, 0.3235257211745479, 0.322418881368486, 0.32124317253071316, 0.3199758654694576, 0.3185528417660627, 0.3169789627525247, 0.31519777015619127, 0.31330228798197146, 0.31115711172476546, 0.3090097988186795, 0.30686837335610034, 0.3043417123307964, 0.30229023799276367, 0.3012353656834662, 0.3030693985172813, 0.3154156840249814, 0.3607916089516997, 0.4149528397867256, 0.4445532812058616, 0.4592009321001898, 0.46991288962243427, 0.4835343115664656, 0.5019139852764364, 0.5174767223323364, 0.5324219690427427, 0.541260839545856, 0.5506421686238534, 0.5649019005861665, 0.5763049070535856, 0.5805518485361362, 0.5837255170418572, 0.5892727060802492, 0.5992091996838985, 0.6072700505352148, 0.6130401753928495, 0.6185300468179706, 0.6222583090120138, 0.6287746182501158, 0.6396883523605627, 0.6479091654170639, 0.6548038645352193, 0.6666852531801868, 0.6767994385506683, 0.6859650850939683, 0.6947898500273242, 0.7001774730389798, 0.7013868354237273, 0.7027622770461875, 0.7039235903783801, 0.7051456077851957, 0.7075218480464093, 0.7115118146759669, 0.7162855090886385, 0.7195350363237051, 0.7234973186888319, 0.7286836838175829, 0.7315706923555784, 0.7326419164279813, 0.7344444269059557, 0.7381435017451379, 0.7427919589096175, 0.7456207456017114, 0.7470008419966069, 0.7485305382457034, 0.7505359825217711, 0.7538501340169059, 0.7575826173660134, 0.761894349120681, 0.7659415708643755, 0.7694378051092644, 0.7738443757981833, 0.7847902678788854, 0.7989548920381089, 0.8068268961336539, 0.811720998806265, 0.8172431657324736, 0.8217638032508153, 0.8243459137565781, 0.8264507665759598, 0.8281538841647501, 0.8289201342887792, 0.8290674866129953, 0.8290238713142511, 0.829045642021672, 0.829213833853419, 0.8294122665765469, 0.8297590339320055, 0.8303282959234365, 0.8307612101046168, 0.8309872687162584, 0.8310508766157181, 0.8311526404444715, 0.8312583052721084, 0.8316103865493069, 0.8318772589248815, 0.8321819898922518, 0.8326570770862247, 0.833155706202155, 0.834198532848919, 0.8361724331251044, 0.8387787205326419, 0.8419807737959778, 0.8448835642149576, 0.8457024141405199, 0.8457853667828716, 0.8462714366880213, 0.8469669485708313, 0.8473724157175987, 0.8475708062577422, 0.8475613517760757, 0.8480411563311336, 0.8486611426835862, 0.8490996652945615, 0.8497655105336038, 0.8504699009753925, 0.85154161286929, 0.8531295841311247, 0.8563032095200315, 0.8623688241847279, 0.8704866999187131, 0.877850915093801, 0.8834917868878953, 0.8885016038382444, 0.8905061352868833, 0.8907255537593101, 0.8909461976193023, 0.8915035441730632, 0.8921254602040232, 0.8921805758603685, 0.8918673361724402, 0.8915676436447222, 0.8915735179523385, 0.8922307956647745, 0.8928022443224702, 0.8932127541507715, 0.8936391659249043, 0.8938902209860177, 0.8940673388030697, 0.8943516339805899, 0.8945886446265214, 0.8949101394575718, 0.8955752701127611, 0.8966963078160665, 0.8982738372911666, 0.9004224514071824, 0.9041624805350182, 0.9087047920003753, 0.9113436052492194, 0.9128208960465651, 0.915203936236015, 0.9185435632219748, 0.9223858585569866, 0.9269236581015547, 0.9300131502780365, 0.9308721049677431, 0.931022966553866, 0.9310216365738446, 0.930987605746363, 0.930995433207756, 0.9309932593065964, 0.9309989867588689, 0.931043347741609, 0.9310865207714607, 0.9311190827113365, 0.9311540952961661]
p_analytic_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_212_stored = [0.3333335001667918, 0.3333334841703318, 0.33333343603948484, 0.33333335534860326, 0.3333332413841125, 0.33333309313822623, 0.33333290930006804, 0.3333326882441153, 0.3333324280158607, 0.3333321263145568, 0.33333178047288414, 0.3333313874333565, 0.33333094372124067, 0.33333044541374207, 0.3333298881051677, 0.3333292668677418, 0.33332857620770806, 0.33332781001630735, 0.3333269615151708, 0.33332602319561544, 0.3333249867512684, 0.3333238430033868, 0.3333225818181617, 0.33332119201522514, 0.3333196612664867, 0.333317975984336, 0.3333161211981414, 0.3333140804178589, 0.33331183548344123, 0.333309366398592, 0.33330665114725966, 0.3333036654910869, 0.3333003827458485, 0.33329677353469145, 0.3332928055157656, 0.333288443081566, 0.3332836470270285, 0.33327837418309636, 0.3332725770121266, 0.33326620316111183, 0.33325919496825923, 0.3332514889179866, 0.3332430150388642, 0.3332336962384312, 0.3332234475681625, 0.33321217541112846, 0.33319977658407374, 0.33318613734474345, 0.3331711322942784, 0.33315462316338285, 0.3331364574697344, 0.3331164670327183, 0.3330944663300412, 0.33307025067906915, 0.333043594223837, 0.33301424770656607, 0.33298193600017556, 0.33294635537566175, 0.3329071704753078, 0.33286401095945695, 0.33281646779098034, 0.3327640891175808, 0.3327063757076257, 0.33264277589028035, 0.3325726799452503, 0.3324954138813901, 0.3324102325367662, 0.3323163119253966, 0.3322127407478252, 0.3320985109738665, 0.33197250739629697, 0.3318334960439837, 0.33168011133203074, 0.33151084181514673, 0.33132401439890075, 0.3311177768523294, 0.3308900784552656, 0.3306386486060123, 0.3303609732114472, 0.3300542686851901, 0.32971545339445435, 0.3293411164292555, 0.3289274836288787, 0.328470380905382, 0.3279651950762704, 0.3274068326953552, 0.3267896778102876, 0.32610755026943633, 0.32535366729768883, 0.3245206128017896, 0.323600321651318, 0.32258409068795535, 0.32146263561146093, 0.32022622524182415, 0.3188649457184913, 0.31736918400953823, 0.31573048634215456, 0.31394307050739073, 0.31200651018046055, 0.30993059631729836, 0.3077444320033026, 0.30551425813066635, 0.3033807214235643, 0.30164414940832995, 0.3009870603268964, 0.30318928773938136, 0.31457254857132905, 0.3698513106744584, 0.39939495538372527, 0.4208545258781095, 0.4389654491561266, 0.45503170614317257, 0.4696505437641434, 0.4831583368260397, 0.49576850304561043, 0.5076275446236284, 0.518841856299449, 0.5294920519966053, 0.5396412669346529, 0.5493402775280881, 0.5586308174605366, 0.5675478143042283, 0.5761209489339945, 0.5843757739352092, 0.5923345358756703, 0.600016792294252, 0.6074398835464428, 0.614619299730829, 0.6215689696753025, 0.6283014925276198, 0.6348283244106708, 0.6411599315381966, 0.6473059163101776, 0.6532751223322006, 0.6590757230032905, 0.6647152959689813, 0.6702008870863575, 0.6755390649401664, 0.6807359680431266, 0.685797345960717, 0.6907285948530941, 0.6955347890597163, 0.7002207086259864, 0.704790863727414, 0.709249516522164, 0.7136007005587626, 0.717848237956369, 0.7219957552674428, 0.7260466973100951, 0.730004340021069, 0.7338718019615618, 0.7376520547578475, 0.741347932663717, 0.7449621412593401, 0.7484972655779268, 0.7519557772224539, 0.7553400412973428, 0.7586523225385777, 0.7618947910545576, 0.7650695277167726, 0.7681785289156751, 0.771223711277841, 0.7742069158833247, 0.7771299121204943, 0.7799944014795411, 0.7828020209188232, 0.7855543461427559, 0.7882528945525505, 0.7908991280405997, 0.7934944557365813, 0.7960402363570445, 0.7985377806381738, 0.8009883535286374, 0.8033931761603241, 0.8057534279367164, 0.8080702482052372, 0.8103447381533713, 0.8125779623627047, 0.8147709503681186, 0.8169246981929251, 0.8190401696111771, 0.8211182975982944, 0.8231599855185, 0.8251661083270179, 0.827137513699742, 0.8290750231070635, 0.8309794328352347, 0.8328515149416306, 0.8346920182642105, 0.8365016691357117, 0.8382811723920387, 0.8400312120821907, 0.8417524522791601, 0.8434455377862861, 0.8451110948028077, 0.8467497316497141, 0.8483620394259136, 0.8499485924876754, 0.8515099491964553, 0.8530466523646855, 0.8545592298419635, 0.8560481949739599, 0.8575140472194928, 0.8589572724285777, 0.8603783434645937, 0.8617777205560136, 0.8631558517115508, 0.8645131732046425, 0.8658501097747312, 0.8671670752156528, 0.8684644725674188, 0.869742694513506, 0.8710021237476573, 0.8722431332076148, 0.8734660864263312, 0.8746713378291187, 0.8758592330127849, 0.8770301089621981, 0.8781842944141817, 0.879322109971411, 0.880443868514405, 0.8815498752386376, 0.8826404280957587, 0.8837158177882123, 0.8847763281368857, 0.8858222362751498, 0.8868538127068422, 0.8878713216721088, 0.888875021215303, 0.8898651633756595, 0.8908419943874772, 0.8918057548600413, 0.8927566798390882, 0.8936949990960839, 0.8946209371495691, 0.8955347134867095, 0.8964365427093033, 0.897326634571719, 0.8982051942513591, 0.8990724223478707, 0.8999285150672252, 0.9007736643388232, 0.9016080579136508, 0.9024318794797638, 0.9032453087690697, 0.9040485216508699, 0.9048416902817003, 0.9056249830935535, 0.9063985650312111, 0.9071625975137676, 0.9079172386162548, 0.9086626430804998, 0.9093989624806526, 0.9101263452248605, 0.9108449366853398]

r_over_R_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_92_stored = [0.0, 0.009375, 0.01875, 0.028125, 0.0375, 0.046875, 0.05625, 0.065625, 0.075, 0.084375, 0.09375, 0.103125, 0.1125, 0.121875, 0.13125, 0.140625, 0.15, 0.159375, 0.16875, 0.178125, 0.1875, 0.196875, 0.20625, 0.215625, 0.225, 0.234375, 0.24375, 0.253125, 0.2625, 0.271875, 0.28125, 0.290625, 0.3, 0.309375, 0.31875, 0.328125, 0.3375, 0.346875, 0.35625, 0.365625, 0.375, 0.384375, 0.39375, 0.403125, 0.4125, 0.421875, 0.43125, 0.440625, 0.45, 0.459375, 0.46875, 0.478125, 0.4875, 0.496875, 0.50625, 0.515625, 0.525, 0.534375, 0.54375, 0.553125, 0.5625, 0.571875, 0.58125, 0.590625, 0.6, 0.609375, 0.61875, 0.628125, 0.6375, 0.646875, 0.65625, 0.665625, 0.675, 0.684375, 0.69375, 0.703125, 0.7125, 0.721875, 0.73125, 0.740625, 0.75, 0.759375, 0.76875, 0.778125, 0.7875, 0.796875, 0.80625, 0.815625, 0.825, 0.834375, 0.84375, 0.853125, 0.8625, 0.871875, 0.88125, 0.890625, 0.9, 0.909375, 0.91875, 0.928125, 0.9375, 0.946875, 0.95625, 0.965625, 0.975, 0.984375, 0.99375, 1.003125, 1.0125, 1.021875, 1.03125, 1.040625, 1.05, 1.059375, 1.06875, 1.078125, 1.0875, 1.096875, 1.10625, 1.115625, 1.125, 1.134375, 1.14375, 1.153125, 1.1625, 1.171875, 1.18125, 1.190625, 1.2, 1.209375, 1.21875, 1.228125, 1.2375, 1.246875, 1.25625, 1.265625, 1.275, 1.284375, 1.29375, 1.303125, 1.3125, 1.321875, 1.33125, 1.340625, 1.35, 1.359375, 1.36875, 1.378125, 1.3875, 1.396875, 1.40625, 1.415625, 1.425, 1.434375, 1.44375, 1.453125, 1.4625, 1.471875, 1.48125, 1.490625, 1.5, 1.509375, 1.51875, 1.528125, 1.5375, 1.546875, 1.55625, 1.565625, 1.575, 1.584375, 1.59375, 1.603125, 1.6125, 1.621875, 1.63125, 1.640625, 1.65, 1.659375, 1.66875, 1.678125, 1.6875, 1.696875, 1.70625, 1.715625, 1.725, 1.734375, 1.74375, 1.753125, 1.7625, 1.771875, 1.78125, 1.790625, 1.8, 1.809375, 1.81875, 1.828125, 1.8375, 1.846875, 1.85625, 1.865625, 1.875, 1.884375, 1.89375, 1.903125, 1.9125, 1.921875, 1.93125, 1.940625, 1.95, 1.959375, 1.96875, 1.978125, 1.9875, 1.996875, 2.00625, 2.015625, 2.025, 2.034375, 2.04375, 2.053125, 2.0625, 2.071875, 2.08125, 2.090625, 2.1, 2.109375, 2.11875, 2.128125, 2.1375, 2.146875, 2.15625, 2.165625, 2.175, 2.184375, 2.19375, 2.203125, 2.2125, 2.221875, 2.23125, 2.240625, 2.25, 2.259375, 2.26875, 2.278125, 2.2875, 2.296875, 2.30625, 2.315625, 2.325, 2.334375, 2.34375, 2.353125, 2.3625, 2.371875, 2.38125]
f_simulation_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_92_stored = [1.5015850881906877e-14, 1.5938140884028122e-06, 3.226728172120821e-06, 4.860022660853569e-06, 6.48986687482733e-06, 8.077029600758431e-06, 9.723151404464397e-06, 1.1514950858301693e-05, 1.3410527425946691e-05, 1.5361413976296152e-05, 1.7352209950203357e-05, 1.94699904468187e-05, 2.176062442389347e-05, 2.416301331131989e-05, 2.6712240307499253e-05, 2.9539496550776167e-05, 3.2666962405331916e-05, 3.5855865651793047e-05, 3.919290864146753e-05, 4.281105705972652e-05, 4.6750928261970285e-05, 5.104102734281092e-05, 5.574915371389521e-05, 6.08199986685826e-05, 6.655265483530447e-05, 7.278719234623735e-05, 7.915074151535762e-05, 8.567663229291694e-05, 9.295935804576706e-05, 0.00010115473268044203, 0.00011048216514799434, 0.00012041765317893406, 0.0001310325581052661, 0.00014304993078153837, 0.00015686859110350892, 0.00017163868704451882, 0.0001874352581505588, 0.0002041911772637388, 0.00022107228546119282, 0.0002397503569794625, 0.0002618331140865135, 0.0002864947558359877, 0.0003132948342542778, 0.0003427643821922799, 0.00037530550267993797, 0.00041078801950986224, 0.00044923957303845386, 0.0004928651605801158, 0.0005392134907711119, 0.0005871197659565814, 0.0006414380452376367, 0.0007008706174185629, 0.0007688306205647258, 0.0008433810979465851, 0.0009250714324012513, 0.001017752001937837, 0.0011185390438160283, 0.0012280521045235837, 0.0013518282298579536, 0.001486116216160194, 0.0016288851410405983, 0.0017850319980597619, 0.001957534638449692, 0.0021430511799018837, 0.002355858558682535, 0.0026007061528862704, 0.00286913959057248, 0.00316605165406654, 0.0035115173056366376, 0.0038912130588239276, 0.004287630120780194, 0.004735693163153094, 0.005257507827515657, 0.0058271082869212195, 0.006427789250682337, 0.007088143422227941, 0.007826289805448918, 0.008634697204766463, 0.009552667842018539, 0.01060060624153519, 0.011759415499336798, 0.013066010734329921, 0.014497870214687594, 0.01606584886320485, 0.017945913698625162, 0.02007591765498022, 0.022440482381963632, 0.025067744098914088, 0.028047847940537076, 0.03141847909662882, 0.03539795068581468, 0.039895999782586324, 0.044927755642881324, 0.05093598144370762, 0.057429874708776786, 0.06484630659875193, 0.07393131904430851, 0.08441409899258202, 0.09740238144842295, 0.11335075263055576, 0.13256801691358686, 0.15507228775290177, 0.18462121420034874, 0.22558541322421122, 0.2850782089981023, 0.3825602701335963, 0.5448136501009225, 0.6132683567189005, 0.6253370929596764, 0.6324223864171473, 0.6454281892164283, 0.6538135160202854, 0.6621797680431554, 0.6734079647960055, 0.7016976615829335, 0.7379331455494542, 0.7521371782119438, 0.7526958156982315, 0.7547871187362979, 0.7595324510704654, 0.768963354818398, 0.774567687624803, 0.7773812092260171, 0.7803310974931833, 0.7881061383517554, 0.7947022860110472, 0.7956472006963872, 0.7965903493228673, 0.7969317773891134, 0.7969679873770963, 0.7975774645187691, 0.7973442591849776, 0.7981507005808244, 0.7995555064521263, 0.8011099617807376, 0.8041347941140657, 0.8099927724796148, 0.8151676063897506, 0.8171354048483312, 0.8181198901365898, 0.8208900437275701, 0.8251036184339525, 0.8310392662270096, 0.8417967066889323, 0.851738557120167, 0.8541881789168991, 0.8548181291252184, 0.8555968711051789, 0.8574697939000924, 0.863101904019469, 0.870663050353866, 0.8744385860432989, 0.8746525676234298, 0.8751161568335968, 0.8754458982425138, 0.8756162101084383, 0.8761012019481255, 0.8771097245529522, 0.8772859939111679, 0.8771299887407922, 0.8782638127504501, 0.8801647615705139, 0.8833552997538154, 0.8890324816341679, 0.8960021416821234, 0.8993002486347002, 0.9019434658879426, 0.9068763710300004, 0.9132406112695264, 0.9184408986087862, 0.9206874235065535, 0.9207643051540313, 0.9208724667730868, 0.9209395124608081, 0.9209730103778194, 0.9209021021454035, 0.9209050687095083, 0.9209815903199782, 0.9209758598815115, 0.9210840986825535, 0.9211622790753542, 0.9211242630367517, 0.9210884805787318, 0.9213116239289634, 0.9216201563338876, 0.921820341062445, 0.9218282644482261, 0.9216810645169976, 0.9215182501118422, 0.9214710524617062, 0.9216522179125952, 0.9219514549651588, 0.9221927611757009, 0.9222067725033536, 0.9220414746515868, 0.9220729330127457, 0.9223518585527379, 0.922561221274172, 0.9226554778746743, 0.9227285744469728, 0.923050836374733, 0.9235326453126856, 0.9239293285271434, 0.9244276309122657, 0.9250495879088443, 0.9258808701937792, 0.9272747742244462, 0.9297589620514685, 0.934125549667016, 0.9381001268261548, 0.9395915804388053, 0.9398747941270497, 0.9398721561924599, 0.9398658009239905, 0.9399277861567068, 0.9399270818787457, 0.9398755042422865, 0.9398636718866736, 0.9399828161681196, 0.9401553296867237, 0.9401622899960905, 0.9399857108852246, 0.9398603671575373, 0.9399780754763958, 0.9400390948843871, 0.9400186559906779, 0.9399932947417146, 0.9400914345076875, 0.9402176177486008, 0.9402601111658647, 0.9401670161168723, 0.940080224904243, 0.9401167476292545, 0.9402712126649525, 0.9403445295760592, 0.9402639249746343, 0.9401609011894998, 0.9401909030555401, 0.9403415465433558, 0.9404930061917575, 0.9405551438997378, 0.9405828023924905, 0.9405570587504541, 0.9404274353022732, 0.9403690739732923, 0.940382721696208, 0.9405020695736482, 0.9406791912516003, 0.9407898098839622, 0.9408278079296113, 0.9408636974972403, 0.9409321774003729, 0.9409055084862026, 0.9408903801291983, 0.9409653371145064]
f_analytic_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_92_stored = [5.55136761539776e-17, 1.4226453844488892e-06, 2.854299149758322e-06, 4.3040398677015174e-06, 5.781087115286238e-06, 7.294873539634997e-06, 8.855118816019634e-06, 1.0471906158919991e-05, 1.2155762070824759e-05, 1.3917740047695134e-05, 1.5769508997105106e-05, 1.7723447175192477e-05, 1.979274250018932e-05, 2.1991500170138952e-05, 2.4334858579141564e-05, 2.683911461817557e-05, 2.9521859537140333e-05, 3.2402126654953777e-05, 3.550055232748126e-05, 3.883955171910129e-05, 4.24435110781892e-05, 4.633899839095476e-05, 5.055499448072776e-05, 5.5123146837194206e-05, 6.007804870350675e-05, 6.545754621979247e-05, 7.130307672753957e-05, 7.766004167775974e-05, 8.457821796792235e-05, 9.211221195757216e-05, 0.00010032196089257351, 0.00010927328699863067, 0.0001190385101080526, 0.0001296971253534113, 0.00014133655321692866, 0.00015405297007176002, 0.00016795222830156553, 0.0001831508761571072, 0.00019977728871185515, 0.00021797292263442769, 0.00023789370902899825, 0.0002597116003161396, 0.0002836162890818507, 0.0003098171190239678, 0.0003385452106251175, 0.00037005582701069104, 0.0004046310086625377, 0.00044258250931061567, 0.0004842550694792665, 0.0005300300689007807, 0.0005803296044128496, 0.0006356210461374085, 0.00069642213181683, 0.0007633066673045405, 0.0008369109105423441, 0.000917940727114868, 0.0010071796178843578, 0.0011054977335754898, 0.0012138620078409488, 0.0013333475597085895, 0.0014651505388944656, 0.0016106026138662558, 0.0017711873334976072, 0.0019485586295632463, 0.002144561770291207, 0.0023612571260683774, 0.002600947168866937, 0.002866207199110918, 0.003159920380135072, 0.003485317764386654, 0.0038460241212033832, 0.004246110528633489, 0.004690154878046973, 0.005183311668848142, 0.00573139275261159, 0.006340961035976801, 0.00701943958876066, 0.00777523915327358, 0.008617907746470895, 0.009558306933720359, 0.010608820493209383, 0.011783602667815068, 0.013098875133562787, 0.014573284364412607, 0.01622833447410945, 0.01808891520024694, 0.020183950947975655, 0.02254720544603093, 0.025218288658574863, 0.028243929779589675, 0.031679604958012766, 0.035591644903829996, 0.0400600022754023, 0.045181942690287885, 0.05107705509568965, 0.05789419027645254, 0.06582129135951013, 0.07509969366982727, 0.08604557643100985, 0.0990833396806949, 0.11479987462322089, 0.1340377365488795, 0.15806654726438138, 0.18892846420318454, 0.2302295568680517, 0.2893480965757367, 0.386376637019887, 0.5493835073960827, 0.5845378604840445, 0.6078416621161116, 0.626370888316594, 0.6420689210111367, 0.6558218335509317, 0.668125536139601, 0.6792921792610009, 0.6895340315554566, 0.6990036012043547, 0.7078150733665282, 0.7160567286913022, 0.7237985973154231, 0.7310974128913053, 0.7379999503928002, 0.7445453504877032, 0.7507667841402815, 0.7566926739928154, 0.7623476090131328, 0.7677530421945464, 0.7729278314578334, 0.7778886646194654, 0.7826503982762003, 0.7872263303397163, 0.7916284222552058, 0.79586748138606, 0.7999533121767038, 0.8038948427041578, 0.8077002308216117, 0.8113769544347225, 0.8149318882213095, 0.8183713695167056, 0.8217012552149952, 0.8249269708430669, 0.8280535535549015, 0.8310856895267152, 0.8340277468339151, 0.8368838045266594, 0.8396576782967587, 0.8423529431397403, 0.8449729538118492, 0.8475208628121339, 0.8499996367154934, 0.8524120707553602, 0.8547608019494454, 0.8570483209834011, 0.8592769829456988, 0.861449017166822, 0.8635665359561419, 0.8656315428002519, 0.8676459396902758, 0.86961153387412, 0.8715300440929749, 0.8734031061620973, 0.8752322782855431, 0.8770190458500337, 0.8787648258045205, 0.880470970825347, 0.882138773064978, 0.8837694677026519, 0.8853642361684176, 0.8869242091549948, 0.8884504694918003, 0.8899440546856738, 0.8914059594203257, 0.8928371378317311, 0.8942385055781367, 0.8956109419096066, 0.8969552914313924, 0.8982723659395674, 0.8995629460041125, 0.9008277825061798, 0.9020675981159711, 0.9032830885715429, 0.9044749240277881, 0.9056437502182488, 0.9067901895928038, 0.9079148423834069, 0.909018287609802, 0.9101010840292565, 0.911163771024533, 0.9122068695001619, 0.9132308825903072, 0.9142362965143025, 0.9152235812448821, 0.9161931912169518, 0.9171455659616641, 0.9180811307031048, 0.9190002969753333, 0.9199034631874081, 0.9207910150776087, 0.9216633263070833, 0.9225207588738824, 0.9233636635960156, 0.9241923805143687, 0.9250072393752498, 0.925808559916222, 0.9265966523360569, 0.9273718176071983, 0.9281343478170406, 0.9288845265413425, 0.929622629055157, 0.9303489227524407, 0.9310636673412365, 0.9317671151476852, 0.9324595113985146, 0.9331410944260201, 0.9338120959337229, 0.9344727412284164, 0.9351232494387307, 0.9357638336960733, 0.9363947013930544, 0.9370160543020428, 0.9376280888539138, 0.9382309962092267, 0.9388249625475151, 0.9394101691104987, 0.9399867924464557, 0.9405550045576676, 0.9411149729719746, 0.9416668609797317, 0.9422108277069742, 0.9427470282536156, 0.9432756138350941, 0.9437967319114089, 0.9443105262506188, 0.9448171371141588, 0.945316701295922, 0.9458093522683008, 0.946295220286128, 0.9467744324312597, 0.9472471127813786, 0.9477133824315959, 0.9481733596146634, 0.94862715978414, 0.9490748956864895, 0.9495166774416431, 0.9499526126180692, 0.950382806299971, 0.9508073611838953, 0.9512263775880764, 0.9516399535945005, 0.9520481850426744, 0.9524511656405685, 0.9528489869836524, 0.9532417386560135, 0.9536295082434366, 0.9540123814145287]
p_simulation_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_92_stored = [0.3359205676015118, 0.3359205577584002, 0.33592051246489585, 0.3359204042723513, 0.33592026550408155, 0.33592012819993455, 0.3359199512538485, 0.33591969513663994, 0.3359193991127014, 0.3359190652646839, 0.33591865461820036, 0.3359182301001724, 0.33591776954626135, 0.3359172585146298, 0.3359166684765692, 0.33591595138718916, 0.3359151072385043, 0.3359142187247872, 0.33591334277447554, 0.3359123567339388, 0.33591124572475933, 0.33591001487878136, 0.33590863982418095, 0.3359070374802931, 0.33590528663150937, 0.33590334538326433, 0.3359012941899983, 0.3358992885551956, 0.3358970351795128, 0.33589435235123594, 0.33589132277982287, 0.33588801505139804, 0.33588432248707034, 0.3358801053597652, 0.3358753749764489, 0.3358703369310942, 0.33586496250282233, 0.335859222174342, 0.33585349385086755, 0.3358472548803743, 0.33583975753977174, 0.3358309388569746, 0.3358213398935142, 0.33581090018235243, 0.33579921274219415, 0.33578619731561, 0.33577232414517116, 0.3357570029870853, 0.3357406441428421, 0.3357239756792542, 0.3357052635903935, 0.33568336077838573, 0.33565874769491816, 0.3356319063731349, 0.33560225278042405, 0.33556927624919736, 0.33553324252854083, 0.33549324181840456, 0.33544881578359126, 0.3354006836993048, 0.335348847920676, 0.33529343428692593, 0.33523451443113206, 0.33516781674065604, 0.33509410082926566, 0.3350094581283422, 0.3349147937705475, 0.33480786396656764, 0.33468518796752256, 0.33455245909082015, 0.33441261519411386, 0.3342544907529573, 0.3340761447087257, 0.3338845700895891, 0.3336825477315305, 0.33346480444095056, 0.3332255226131199, 0.33295882810544014, 0.33266715663719026, 0.3323467026632088, 0.33198667309265184, 0.3315861849602692, 0.33114764060879825, 0.3306572141950777, 0.3301146922120356, 0.32948423236564417, 0.3288071999340224, 0.32807163617217505, 0.3272804718303042, 0.32634345740647996, 0.32531870089352255, 0.3241962704898327, 0.3229580176350007, 0.3217187595764181, 0.3202059943452772, 0.31862349718230254, 0.3169959552120667, 0.31494115385564153, 0.31283207951522995, 0.310681750246859, 0.308349457689527, 0.3056258827114479, 0.3034372290000216, 0.30233238319028427, 0.3049446785961443, 0.3205271076559085, 0.3855936528016756, 0.42777091076063367, 0.4389639279856729, 0.44627540186842474, 0.4610860837193349, 0.4701421135574183, 0.4801092702721168, 0.493369782658975, 0.5269055927643012, 0.5706495794870315, 0.5882458350611381, 0.5888089591656301, 0.5913656834816959, 0.597667221040021, 0.6100665322875326, 0.6175218069130836, 0.6214018138008137, 0.6253116550543696, 0.6359939100436514, 0.6451519948476712, 0.6465863145336793, 0.6480874929453925, 0.6485187188139135, 0.6484813588646366, 0.6494345428624693, 0.649131695436975, 0.6503573386398193, 0.6524692504053705, 0.6548555380207837, 0.659531409567591, 0.6684610835353141, 0.6763263825716805, 0.6793060341557652, 0.6808244015536219, 0.6852604852885691, 0.691949694464646, 0.7013166645618073, 0.7182603980988544, 0.7339134499134556, 0.7378080100733172, 0.7388390626234149, 0.7400979005711089, 0.7430967489153795, 0.7522116707172564, 0.7644999240054939, 0.7706629560800438, 0.7710066026784352, 0.7717942366635717, 0.7723762915088775, 0.7726482130894408, 0.773450771688118, 0.7751765231554864, 0.7755038219554958, 0.7752506858444016, 0.7771609689340364, 0.780379234576242, 0.7857949643700771, 0.7954079372241172, 0.8072017954473102, 0.812800657272706, 0.8173402701388162, 0.8257984629989739, 0.8366977123162271, 0.8455844384532669, 0.849405787147615, 0.8495275476597676, 0.849717365953765, 0.8498417321442029, 0.8499122525186246, 0.849792848426423, 0.8498069273195047, 0.8499451575614319, 0.8499254046264386, 0.8501154006850831, 0.850257200921911, 0.8501974052666306, 0.8501404703571187, 0.8505471975204096, 0.8510958753569527, 0.8514533623772759, 0.8514695387269992, 0.8512053046953779, 0.8509126005615725, 0.8508352865740026, 0.8511630193151692, 0.8517006206651926, 0.8521420816799565, 0.8521777207957699, 0.851872483310978, 0.8519202352541952, 0.8524192772901449, 0.8528024633544873, 0.8529774790556414, 0.8531198811685854, 0.8537025800642467, 0.8545735631810117, 0.8552926886561231, 0.8562016084044818, 0.8573270145157739, 0.8588307054883259, 0.8613584469646527, 0.8658615938365646, 0.8737672766888108, 0.8809665154150058, 0.8836770960644278, 0.8841916708136459, 0.8841819456886079, 0.884168289368664, 0.8842869760810089, 0.8842860900645282, 0.8841905117906236, 0.8841704713038963, 0.8843987524757065, 0.8847270038963762, 0.8847392840262663, 0.8844040107274743, 0.8841650088459885, 0.8843888891775077, 0.8845034061459675, 0.8844642429947164, 0.8844182151627613, 0.8846065741102417, 0.884846604124988, 0.8849259889306675, 0.8847493585148297, 0.8845848016267136, 0.8846543759794678, 0.8849479463428662, 0.8850872981123769, 0.8849354844350077, 0.8847389377962137, 0.8847953215890378, 0.8850824553029527, 0.8853739905203729, 0.8854961307425046, 0.8855515232315245, 0.8855025536614054, 0.8852515360226653, 0.8851355715951816, 0.8851592708568388, 0.8853902069355006, 0.8857303527760603, 0.885941245642585, 0.8860125466953278, 0.8860822413144843, 0.8862162718525758, 0.8861677593688312, 0.8861399522172607, 0.8862808343993129]
p_analytic_radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_92_stored = [0.3333335001667918, 0.3333334841703318, 0.33333343603948484, 0.33333335534860326, 0.3333332413841125, 0.33333309313822623, 0.33333290930006804, 0.3333326882441153, 0.3333324280158607, 0.3333321263145568, 0.33333178047288414, 0.3333313874333565, 0.33333094372124067, 0.33333044541374207, 0.3333298881051677, 0.3333292668677418, 0.33332857620770806, 0.33332781001630735, 0.3333269615151708, 0.33332602319561544, 0.3333249867512684, 0.3333238430033868, 0.3333225818181617, 0.33332119201522514, 0.3333196612664867, 0.333317975984336, 0.3333161211981414, 0.3333140804178589, 0.33331183548344123, 0.333309366398592, 0.33330665114725966, 0.3333036654910869, 0.3333003827458485, 0.33329677353469145, 0.3332928055157656, 0.333288443081566, 0.3332836470270285, 0.33327837418309636, 0.3332725770121266, 0.33326620316111183, 0.33325919496825923, 0.3332514889179866, 0.3332430150388642, 0.3332336962384312, 0.3332234475681625, 0.33321217541112846, 0.33319977658407374, 0.33318613734474345, 0.3331711322942784, 0.33315462316338285, 0.3331364574697344, 0.3331164670327183, 0.3330944663300412, 0.33307025067906915, 0.333043594223837, 0.33301424770656607, 0.33298193600017556, 0.33294635537566175, 0.3329071704753078, 0.33286401095945695, 0.33281646779098034, 0.3327640891175808, 0.3327063757076257, 0.33264277589028035, 0.3325726799452503, 0.3324954138813901, 0.3324102325367662, 0.3323163119253966, 0.3322127407478252, 0.3320985109738665, 0.33197250739629697, 0.3318334960439837, 0.33168011133203074, 0.33151084181514673, 0.33132401439890075, 0.3311177768523294, 0.3308900784552656, 0.3306386486060123, 0.3303609732114472, 0.3300542686851901, 0.32971545339445435, 0.3293411164292555, 0.3289274836288787, 0.328470380905382, 0.3279651950762704, 0.3274068326953552, 0.3267896778102876, 0.32610755026943633, 0.32535366729768883, 0.3245206128017896, 0.323600321651318, 0.32258409068795535, 0.32146263561146093, 0.32022622524182415, 0.3188649457184913, 0.31736918400953823, 0.31573048634215456, 0.31394307050739073, 0.31200651018046055, 0.30993059631729836, 0.3077444320033026, 0.30551425813066635, 0.3033807214235643, 0.30164414940832995, 0.3009870603268964, 0.30318928773938136, 0.31457254857132905, 0.3698513106744584, 0.39939495538372527, 0.4208545258781095, 0.4389654491561266, 0.45503170614317257, 0.4696505437641434, 0.4831583368260397, 0.49576850304561043, 0.5076275446236284, 0.518841856299449, 0.5294920519966053, 0.5396412669346529, 0.5493402775280881, 0.5586308174605366, 0.5675478143042283, 0.5761209489339945, 0.5843757739352092, 0.5923345358756703, 0.600016792294252, 0.6074398835464428, 0.614619299730829, 0.6215689696753025, 0.6283014925276198, 0.6348283244106708, 0.6411599315381966, 0.6473059163101776, 0.6532751223322006, 0.6590757230032905, 0.6647152959689813, 0.6702008870863575, 0.6755390649401664, 0.6807359680431266, 0.685797345960717, 0.6907285948530941, 0.6955347890597163, 0.7002207086259864, 0.704790863727414, 0.709249516522164, 0.7136007005587626, 0.717848237956369, 0.7219957552674428, 0.7260466973100951, 0.730004340021069, 0.7338718019615618, 0.7376520547578475, 0.741347932663717, 0.7449621412593401, 0.7484972655779268, 0.7519557772224539, 0.7553400412973428, 0.7586523225385777, 0.7618947910545576, 0.7650695277167726, 0.7681785289156751, 0.771223711277841, 0.7742069158833247, 0.7771299121204943, 0.7799944014795411, 0.7828020209188232, 0.7855543461427559, 0.7882528945525505, 0.7908991280405997, 0.7934944557365813, 0.7960402363570445, 0.7985377806381738, 0.8009883535286374, 0.8033931761603241, 0.8057534279367164, 0.8080702482052372, 0.8103447381533713, 0.8125779623627047, 0.8147709503681186, 0.8169246981929251, 0.8190401696111771, 0.8211182975982944, 0.8231599855185, 0.8251661083270179, 0.827137513699742, 0.8290750231070635, 0.8309794328352347, 0.8328515149416306, 0.8346920182642105, 0.8365016691357117, 0.8382811723920387, 0.8400312120821907, 0.8417524522791601, 0.8434455377862861, 0.8451110948028077, 0.8467497316497141, 0.8483620394259136, 0.8499485924876754, 0.8515099491964553, 0.8530466523646855, 0.8545592298419635, 0.8560481949739599, 0.8575140472194928, 0.8589572724285777, 0.8603783434645937, 0.8617777205560136, 0.8631558517115508, 0.8645131732046425, 0.8658501097747312, 0.8671670752156528, 0.8684644725674188, 0.869742694513506, 0.8710021237476573, 0.8722431332076148, 0.8734660864263312, 0.8746713378291187, 0.8758592330127849, 0.8770301089621981, 0.8781842944141817, 0.879322109971411, 0.880443868514405, 0.8815498752386376, 0.8826404280957587, 0.8837158177882123, 0.8847763281368857, 0.8858222362751498, 0.8868538127068422, 0.8878713216721088, 0.888875021215303, 0.8898651633756595, 0.8908419943874772, 0.8918057548600413, 0.8927566798390882, 0.8936949990960839, 0.8946209371495691, 0.8955347134867095, 0.8964365427093033, 0.897326634571719, 0.8982051942513591, 0.8990724223478707, 0.8999285150672252, 0.9007736643388232, 0.9016080579136508, 0.9024318794797638, 0.9032453087690697, 0.9040485216508699, 0.9048416902817003, 0.9056249830935535, 0.9063985650312111, 0.9071625975137676, 0.9079172386162548, 0.9086626430804998, 0.9093989624806526, 0.9101263452248605, 0.9108449366853398]

In [ ]:
runs = [
    (378, "radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_378_stored"),
    (212, "radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_212_stored"),
    (92, "radiating_sphere_test_L_cm_4p212e6_2p808e6_2p808e6_ncell_384_256_256_dts_9p15emin8_deposition_1_bc_periodic_empty_ppepc_92_stored"),
]

def stored(name, suffix):
    return np.asarray(globals()[f"{name}_{suffix}"], dtype=float)

for quantity, sim_name, analytic_name in [
    ("f", "f_simulation", "f_analytic"),
    ("p", "p_simulation", "p_analytic"),
]:
    fig, ax = plt.subplots(figsize=(10, 6))
    for color_index, (n_ppc, suffix) in enumerate(runs):
        ax.plot(
            stored("r_over_R", suffix),
            stored(sim_name, suffix),
            color=f"C{color_index}",
            linestyle="dotted",
            linewidth=4,
            label=rf"$\hat{{\mathrm{{{quantity}}}}}_{{\mathrm{{sim}}}},\, N_{{\mathrm{{ppc}}}} = {n_ppc}$",
        )
    r_fine = stored("r_over_R", runs[0][1])
    ax.plot(
        r_fine,
        stored(analytic_name, runs[0][1]),
        color="k",
        linestyle="solid",
        alpha=0.5,
        linewidth=4,
        label=rf"$\hat{{\mathrm{{{quantity}}}}}_{{\mathrm{{analytic}}}}$",
    )
    # ax.axvline(1.0, color="k", linestyle=":", linewidth=1.5, label=r"$r = R$")
    ax.set_xlabel(r"$r / R$")
    ax.set_ylabel(rf"${quantity}$")
    ax.set_xlim(0.0, None)
    leg = ax.legend(framealpha=0.0, ncol=1, fontsize=16)
    pf.apply_custom_settings(ax, leg, False)
    fig.tight_layout()
    plt.show()
